<a href="https://colab.research.google.com/github/giabq/plnEp2/blob/main/finalModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#TODO: ver sobre interpretabilidade + células textuais finais (comparação com o ep1; rotulação do teste 2) [talvez mais alguma coisa]

#Instalação, imports, configurações e GPU

In [9]:
# @title
# ============================================================
# 1. INSTALAÇÃO
# ============================================================

!pip install -q "transformers>=4.51.0,<5" accelerate sentencepiece safetensors openpyxl

# ============================================================
# 2. IMPORTS
# ============================================================

import os
import gc
import json
import random
import shutil
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

warnings.filterwarnings("ignore")

# ============================================================
# 3. REPRODUTIBILIDADE
# ============================================================

SEED = 42

def set_seed(seed=SEED):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

# ============================================================
# 4. CONFIGURAÇÕES GERAIS
# ============================================================

CLASSES = ["c1", "c234", "c5"]

LABEL_TO_ID = {
    label: index
    for index, label in enumerate(CLASSES)
}

ID_TO_LABEL = {
    index: label
    for label, index in LABEL_TO_ID.items()
}

N_SPLITS = 3

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Diretório de resultados
OUTPUT_DIR = "/content/ensemble_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Device:", DEVICE)

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

    total_memory = (
        torch.cuda.get_device_properties(0).total_memory / 1024**3
    )

    free_memory = torch.cuda.mem_get_info()[0] / 1024**3

    print(f"VRAM total: {total_memory:.2f} GB")
    print(f"VRAM disponível: {free_memory:.2f} GB")

else:

    print("AVISO: nenhuma GPU detectada.")

print("\nClasses:", CLASSES)
print("Diretório de saída:", OUTPUT_DIR)

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM total: 39.49 GB
VRAM disponível: 39.08 GB

Classes: ['c1', 'c234', 'c5']
Diretório de saída: /content/ensemble_outputs


#Repositório, carregamento e validação dos dados

In [14]:
# @title
import os
import subprocess

REPO_DIR = "/content/plnEp2"

print("=== CONTEÚDO DE /content ===")
print(os.listdir("/content"))

print("\n=== CONTEÚDO DE /content/plnEp2 ===")
if os.path.exists(REPO_DIR):
    for root, dirs, files in os.walk(REPO_DIR):
        level = root.replace(REPO_DIR, "").count(os.sep)
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        for file in files[:15]:
            print(f"{indent}  {file}")
        dirs[:] = dirs[:10]
else:
    print("Pasta não encontrada.")

print("\n=== LOCALIZAÇÃO DE train.csv ===")
for base in ["/content", "/content/drive/MyDrive"]:
    if os.path.exists(base):
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in [
                ".git", "node_modules", ".cache"
            ]]
            if "train.csv" in files:
                print(os.path.join(root, "train.csv"))

print("\n=== VERIFICAÇÃO DO GIT ===")
if os.path.exists(REPO_DIR):
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "rev-parse", "--is-inside-work-tree"],
        capture_output=True,
        text=True
    )
    print(result.stdout.strip() or result.stderr.strip())

=== CONTEÚDO DE /content ===
['.config', 'ensemble_outputs', 'plnEp2', 'sample_data']

=== CONTEÚDO DE /content/plnEp2 ===
plnEp2/
  results/

=== LOCALIZAÇÃO DE train.csv ===

=== VERIFICAÇÃO DO GIT ===
fatal: not a git repository (or any of the parent directories): .git


In [15]:
# @title
# ============================================================
# 1. CLONAR / ATUALIZAR REPOSITÓRIO
# ============================================================

import os
import shutil
import subprocess

REPO_URL = "https://github.com/giabq/plnEp2.git"
REPO_DIR = "/content/plnEp2"
TEMP_REPO_DIR = "/content/plnEp2_clone_temp"

if not os.path.exists(REPO_DIR):
    # Caso a pasta não exista, clona diretamente
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

elif os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # Caso já seja um repositório Git, atualiza
    print("Repositório Git encontrado. Atualizando...")

    subprocess.run(
        ["git", "-C", REPO_DIR, "pull", "--ff-only"],
        check=True
    )

else:
    # A pasta existe, mas não é um repositório Git.
    # Preserva seu conteúdo e clona em uma pasta temporária.
    print("Pasta existente não é um repositório Git.")
    print("Clonando o repositório em uma pasta temporária...")

    if os.path.exists(TEMP_REPO_DIR):
        shutil.rmtree(TEMP_REPO_DIR)

    subprocess.run(
        ["git", "clone", REPO_URL, TEMP_REPO_DIR],
        check=True
    )

    # Copia os arquivos do clone para a pasta original.
    # Não sobrescreve arquivos já existentes.
    shutil.copytree(
        TEMP_REPO_DIR,
        REPO_DIR,
        dirs_exist_ok=True
    )

    # Remove somente a cópia temporária.
    shutil.rmtree(TEMP_REPO_DIR)

    print("Repositório restaurado sem apagar os arquivos existentes.")

DATA_DIR = os.path.join(REPO_DIR, "data")

# ============================================================
# 2. CAMINHOS
# ============================================================

TRAIN_RAW_PATH = os.path.join(DATA_DIR, "train.csv")
TRAIN_CLEAN_PATH = os.path.join(DATA_DIR, "train_limpo.csv")

TEST1_RAW_PATH = os.path.join(DATA_DIR, "test1.xlsx")
TEST1_CLEAN_PATH = os.path.join(DATA_DIR, "test1_limpo.csv")

TEST1_LABELS_PATH = os.path.join(
    DATA_DIR,
    "test1-rotulado-para-teste-estatistico.xlsx"
)

# Verifica se os arquivos necessários existem
required_files = [
    TRAIN_RAW_PATH,
    TRAIN_CLEAN_PATH,
    TEST1_RAW_PATH,
    TEST1_CLEAN_PATH,
    TEST1_LABELS_PATH
]

missing_files = [
    path for path in required_files
    if not os.path.isfile(path)
]

if missing_files:
    raise FileNotFoundError(
        "Os seguintes arquivos não foram encontrados:\n"
        + "\n".join(missing_files)
        + "\n\nVerifique se eles estão disponíveis no repositório "
          "ou se precisam ser adicionados manualmente."
    )

# ============================================================
# 3. CARREGAMENTO
# ============================================================

train_raw = pd.read_csv(TRAIN_RAW_PATH)
train_clean = pd.read_csv(TRAIN_CLEAN_PATH)

test1_original = pd.read_excel(TEST1_RAW_PATH)
test1_clean = pd.read_csv(TEST1_CLEAN_PATH)
test1_labels = pd.read_excel(TEST1_LABELS_PATH)

# ============================================================
# 4. VERIFICAÇÃO DAS COLUNAS
# ============================================================

required_columns = {"resp_text", "clarity"}

datasets = {
    "train_raw": train_raw,
    "train_clean": train_clean,
    "test1_original": test1_original,
    "test1_clean": test1_clean,
    "test1_labels": test1_labels
}

for name, df in datasets.items():

    missing = required_columns - set(df.columns)

    if missing:

        raise ValueError(
            f"{name}: colunas ausentes: {missing}"
        )

# ============================================================
# 5. VERIFICAÇÃO DE ALINHAMENTO
# ============================================================

assert len(train_raw) == len(train_clean), \
    "Quantidade de linhas diferente no treinamento."

assert len(test1_original) == len(test1_clean), \
    "Quantidade de linhas diferente no Teste 1 limpo."

assert len(test1_original) == len(test1_labels), \
    "Quantidade de linhas diferente entre Teste 1 e arquivo rotulado."

assert train_raw["clarity"].astype(str).equals(
    train_clean["clarity"].astype(str)
), "Os rótulos do treinamento não estão alinhados."

assert test1_original["resp_text"].fillna("").astype(str).equals(
    test1_labels["resp_text"].fillna("").astype(str)
), "Os textos originais do Teste 1 não estão alinhados."

# ============================================================
# 6. NORMALIZAÇÃO MÍNIMA DOS TEXTOS ORIGINAIS
# ============================================================

def normalize_raw_text(series):

    return (
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .tolist()
    )


X_train_raw = normalize_raw_text(
    train_raw["resp_text"]
)

X_test1_raw = normalize_raw_text(
    test1_original["resp_text"]
)

X_train_clean = normalize_raw_text(
    train_clean["resp_text"]
)

X_test1_clean = normalize_raw_text(
    test1_clean["resp_text"]
)

# ============================================================
# 7. RÓTULOS
# ============================================================

y_train = train_raw["clarity"].astype(str).values

y_test1 = test1_labels["clarity"].astype(str).values

unexpected_train = set(np.unique(y_train)) - set(CLASSES)
unexpected_test = set(np.unique(y_test1)) - set(CLASSES)

assert not unexpected_train, \
    f"Classes inesperadas no treinamento: {unexpected_train}"

assert not unexpected_test, \
    f"Classes inesperadas no Teste 1: {unexpected_test}"

y_train_encoded = np.array([
    LABEL_TO_ID[label]
    for label in y_train
])

y_test1_encoded = np.array([
    LABEL_TO_ID[label]
    for label in y_test1
])

# ============================================================
# 8. VERIFICAÇÃO FINAL
# ============================================================

print("Treinamento:", len(X_train_raw))
print("Teste 1:", len(X_test1_raw))

print("\nDistribuição do treinamento:")
print(pd.Series(y_train).value_counts())

print("\nDistribuição do Teste 1:")
print(pd.Series(y_test1).value_counts())

print("\nComprimento médio dos textos:")

print(
    "Treinamento original:",
    np.mean([len(x) for x in X_train_raw])
)

print(
    "Treinamento limpo:",
    np.mean([len(x) for x in X_train_clean])
)

print("\nDados carregados e validados.")

Pasta existente não é um repositório Git.
Clonando o repositório em uma pasta temporária...
Repositório restaurado sem apagar os arquivos existentes.
Treinamento: 20092
Teste 1: 900

Distribuição do treinamento:
c5      6892
c234    6853
c1      6347
Name: count, dtype: int64

Distribuição do Teste 1:
c5      300
c1      300
c234    300
Name: count, dtype: int64

Comprimento médio dos textos:
Treinamento original: 998.3792056539917
Treinamento limpo: 918.3129603822417

Dados carregados e validados.


#Folds, avaliação e modelos TF-IDF

In [16]:
# @title
# ============================================================
# 1. CRIAÇÃO DOS FOLDS
# ============================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

FOLDS = list(
    skf.split(
        np.zeros(len(y_train_encoded)),
        y_train_encoded
    )
)

print("Folds criados:")

for i, (train_idx, val_idx) in enumerate(FOLDS, start=1):

    print(
        f"Fold {i}: "
        f"{len(train_idx)} treino | "
        f"{len(val_idx)} validação"
    )


# ============================================================
# 2. FUNÇÕES DE AVALIAÇÃO
# ============================================================

def evaluate_predictions(y_true, probabilities, model_name):

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    print("\n" + "=" * 65)
    print("RESULTADOS:", model_name)
    print("=" * 65)

    print(
        "Accuracy:",
        round(accuracy_score(y_true, predictions), 4)
    )

    print(
        "Macro-F1:",
        round(f1_score(
            y_true,
            predictions,
            average="macro"
        ), 4)
    )

    print(
        "Macro-Precision:",
        round(precision_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ), 4)
    )

    print(
        "Macro-Recall:",
        round(recall_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        ), 4)
    )

    print("\nClassification Report:")

    print(
        classification_report(
            y_true,
            predictions,
            labels=[0, 1, 2],
            target_names=CLASSES,
            zero_division=0
        )
    )

    print("Confusion Matrix:")

    print(
        confusion_matrix(
            y_true,
            predictions,
            labels=[0, 1, 2]
        )
    )

    return predictions


# ============================================================
# 3. FUNÇÃO GERAL PARA TF-IDF
# ============================================================

def run_tfidf_model(name, vectorizer, C=1.0):

    print("\n" + "=" * 65)
    print("INICIANDO:", name)
    print("=" * 65)

    oof_path = f"{OUTPUT_DIR}/{name}_oof.npy"
    test_path = f"{OUTPUT_DIR}/{name}_test1.npy"

    # Reaproveitar resultados existentes
    if os.path.exists(oof_path) and os.path.exists(test_path):

        print("Resultados encontrados. Carregando...")

        return (
            np.load(oof_path),
            np.load(test_path)
        )

    pipeline = Pipeline([
        ("tfidf", vectorizer),

        ("lr", LogisticRegression(
            C=C,
            class_weight="balanced",
            max_iter=2000,
            random_state=SEED
        ))
    ])

    # Previsões OOF
    oof_probabilities = cross_val_predict(
        pipeline,
        X_train_clean,
        y_train_encoded,
        cv=FOLDS,
        method="predict_proba",
        n_jobs=1
    )

    # Treinamento final com 100% dos dados
    pipeline.fit(
        X_train_clean,
        y_train_encoded
    )

    # Previsões do Teste 1
    test_probabilities = pipeline.predict_proba(
        X_test1_clean
    )

    # Garantir ordem das classes
    assert np.array_equal(
        pipeline.named_steps["lr"].classes_,
        np.array([0, 1, 2])
    )

    # Salvar resultados
    np.save(oof_path, oof_probabilities)
    np.save(test_path, test_probabilities)

    evaluate_predictions(
        y_test1_encoded,
        test_probabilities,
        name
    )

    del pipeline

    gc.collect()

    return oof_probabilities, test_probabilities


# ============================================================
# 4. MODELO 1 — TF-IDF WORD
# ============================================================

word_oof, word_test = run_tfidf_model(

    name="tfidf_word",

    vectorizer=TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 3),
        sublinear_tf=True,
        lowercase=True,
        use_idf=True,
        smooth_idf=True,
        norm="l2"
    ),

    C=1.0
)


# ============================================================
# 5. MODELO 2 — TF-IDF CHAR
# ============================================================

char_oof, char_test = run_tfidf_model(

    name="tfidf_char",

    vectorizer=TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        sublinear_tf=True,
        lowercase=True,
        use_idf=True,
        smooth_idf=True,
        norm="l2",
        min_df=2,
        max_features=300000
    ),

    C=1.0
)

print("\nModelos TF-IDF concluídos.")

Folds criados:
Fold 1: 13394 treino | 6698 validação
Fold 2: 13395 treino | 6697 validação
Fold 3: 13395 treino | 6697 validação

INICIANDO: tfidf_word

RESULTADOS: tfidf_word
Accuracy: 0.4944
Macro-F1: 0.4938
Macro-Precision: 0.4939
Macro-Recall: 0.4944

Classification Report:
              precision    recall  f1-score   support

          c1       0.51      0.48      0.50       300
        c234       0.45      0.44      0.44       300
          c5       0.52      0.56      0.54       300

    accuracy                           0.49       900
   macro avg       0.49      0.49      0.49       900
weighted avg       0.49      0.49      0.49       900

Confusion Matrix:
[[145  85  70]
 [ 85 132  83]
 [ 53  79 168]]

INICIANDO: tfidf_char

RESULTADOS: tfidf_char
Accuracy: 0.4678
Macro-F1: 0.4661
Macro-Precision: 0.4652
Macro-Recall: 0.4678

Classification Report:
              precision    recall  f1-score   support

          c1       0.49      0.52      0.50       300
        c234     

#BERTimbau: tokenização, treinamento e previsões OOF

In [17]:
# @title
# ============================================================
# 1. CONFIGURAÇÕES DO BERT
# ============================================================

BERT_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

BERT_MAX_LENGTH = 256

BERT_EPOCHS = 3
BERT_LR = 3e-5
BERT_WEIGHT_DECAY = 0.01

BERT_EFFECTIVE_BATCH = 16
BERT_MICRO_BATCH = 8

BERT_ACCUMULATION_STEPS = (
    BERT_EFFECTIVE_BATCH // BERT_MICRO_BATCH
)

BERT_NUM_WORKERS = 0

tokenizer = AutoTokenizer.from_pretrained(
    BERT_MODEL_NAME
)

print("BERT tokenizer carregado.")


# ============================================================
# 2. DATASET
# ============================================================

class TextClassificationDataset(Dataset):

    def __init__(self, encodings, labels=None):

        self.encodings = encodings
        self.labels = labels

    def __len__(self):

        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(value[idx], dtype=torch.long)
            for key, value in self.encodings.items()
        }

        if self.labels is not None:

            item["labels"] = torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )

        return item


# ============================================================
# 3. TOKENIZAÇÃO CONVENCIONAL
# ============================================================

def tokenize_conventional(texts):

    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=BERT_MAX_LENGTH,
        return_attention_mask=True
    )


# ============================================================
# 4. TOKENIZAÇÃO HEAD + TAIL
# ============================================================

def tokenize_head_tail(texts):

    all_input_ids = []
    all_attention_masks = []

    for text in texts:

        # Tokenizar sem tokens especiais
        token_ids = tokenizer.encode(
            text,
            add_special_tokens=False
        )

        # Preservar 127 tokens iniciais e 127 finais
        if len(token_ids) > 254:

            token_ids = (
                token_ids[:127] +
                token_ids[-127:]
            )

        # Inserir CLS e SEP
        input_ids = tokenizer.build_inputs_with_special_tokens(
            token_ids
        )

        # Máscara de atenção
        attention_mask = [1] * len(input_ids)

        # Padding
        padding_length = (
            BERT_MAX_LENGTH - len(input_ids)
        )

        input_ids += (
            [tokenizer.pad_token_id] * padding_length
        )

        attention_mask += (
            [0] * padding_length
        )

        all_input_ids.append(input_ids)
        all_attention_masks.append(attention_mask)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks
    }


# ============================================================
# 5. CRIAÇÃO DO MODELO
# ============================================================

def create_bert_model():

    model = AutoModelForSequenceClassification.from_pretrained(
        BERT_MODEL_NAME,
        num_labels=3,
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID
    )

    return model.to(DEVICE)


# ============================================================
# 6. TREINAMENTO
# ============================================================

def train_bert_model(model, train_loader):

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BERT_LR,
        weight_decay=BERT_WEIGHT_DECAY
    )

    accumulation_steps = BERT_ACCUMULATION_STEPS

    updates_per_epoch = int(
        np.ceil(len(train_loader) / accumulation_steps)
    )

    total_updates = updates_per_epoch * BERT_EPOCHS

    warmup_steps = int(total_updates * 0.1)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_updates
    )

    use_amp = DEVICE.type == "cuda"

    scaler = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    for epoch in range(BERT_EPOCHS):

        model.train()

        total_loss = 0

        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(train_loader):

            batch = {
                key: value.to(DEVICE)
                for key, value in batch.items()
            }

            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):

                outputs = model(**batch)

                loss = outputs.loss

                scaled_loss = (
                    loss / accumulation_steps
                )

            scaler.scale(scaled_loss).backward()

            total_loss += loss.item()

            # Atualizar pesos a cada acumulação
            is_accumulation_step = (
                (step + 1) % accumulation_steps == 0
            )

            is_last_batch = (
                step + 1 == len(train_loader)
            )

            if is_accumulation_step or is_last_batch:

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                scaler.step(optimizer)
                scaler.update()

                scheduler.step()

                optimizer.zero_grad(set_to_none=True)

        average_loss = total_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{BERT_EPOCHS} "
            f"- Loss: {average_loss:.4f}"
        )

    return model


# ============================================================
# 7. INFERÊNCIA
# ============================================================

@torch.no_grad()
def predict_bert(model, encodings, batch_size=32):

    dataset = TextClassificationDataset(encodings)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=BERT_NUM_WORKERS
    )

    model.eval()

    all_probabilities = []

    use_amp = DEVICE.type == "cuda"

    for batch in loader:

        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.cuda.amp.autocast(
            enabled=use_amp
        ):

            outputs = model(**batch)

        probabilities = torch.softmax(
            outputs.logits.float(),
            dim=1
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

    return np.concatenate(
        all_probabilities,
        axis=0
    )


# ============================================================
# 8. TREINAR UM FOLD
# ============================================================

def train_one_bert_fold(
    train_texts,
    train_labels,
    val_texts,
    tokenization_function,
    seed=SEED
):

    set_seed(seed)

    train_encodings = tokenization_function(
        train_texts
    )

    val_encodings = tokenization_function(
        val_texts
    )

    train_dataset = TextClassificationDataset(
        train_encodings,
        train_labels
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BERT_MICRO_BATCH,
        shuffle=True,
        num_workers=BERT_NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    )

    model = create_bert_model()

    model = train_bert_model(
        model,
        train_loader
    )

    val_probabilities = predict_bert(
        model,
        val_encodings
    )

    del model
    del train_dataset
    del train_loader
    del train_encodings
    del val_encodings

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return val_probabilities


# ============================================================
# 9. PIPELINE COMPLETO: OOF + MODELO FINAL
# ============================================================

def run_bert_component(
    name,
    tokenization_function
):

    print("\n" + "=" * 65)
    print("INICIANDO:", name)
    print("=" * 65)

    oof_path = f"{OUTPUT_DIR}/{name}_oof.npy"
    test_path = f"{OUTPUT_DIR}/{name}_test1.npy"

    # Reaproveitar resultados completos
    if os.path.exists(oof_path) and os.path.exists(test_path):

        print("Resultados completos encontrados.")

        return (
            np.load(oof_path),
            np.load(test_path)
        )

    # --------------------------------------------------------
    # PREVISÕES OOF
    # --------------------------------------------------------

    oof_probabilities = np.zeros(
        (len(X_train_raw), 3),
        dtype=np.float32
    )

    for fold_number, (train_idx, val_idx) in enumerate(FOLDS):

        fold_path = (
            f"{OUTPUT_DIR}/{name}_fold_{fold_number + 1}.npy"
        )

        if os.path.exists(fold_path):

            print(
                f"Fold {fold_number + 1}: "
                "carregando previsões salvas."
            )

            val_probabilities = np.load(fold_path)

        else:

            print(
                f"\nTreinando fold "
                f"{fold_number + 1}/{N_SPLITS}"
            )

            train_texts = [
                X_train_raw[i]
                for i in train_idx
            ]

            val_texts = [
                X_train_raw[i]
                for i in val_idx
            ]

            val_probabilities = train_one_bert_fold(
                train_texts,
                y_train_encoded[train_idx],
                val_texts,
                tokenization_function,
                seed=SEED + fold_number
            )

            np.save(
                fold_path,
                val_probabilities
            )

        oof_probabilities[val_idx] = val_probabilities

        del val_probabilities
        gc.collect()

    np.save(
        oof_path,
        oof_probabilities
    )

    # --------------------------------------------------------
    # MODELO FINAL: 100% DO TREINAMENTO
    # --------------------------------------------------------

    if os.path.exists(test_path):

        print("Previsões do Teste 1 já encontradas.")

        test_probabilities = np.load(test_path)

    else:

        print("\nTreinando modelo final com 100% dos dados.")

        set_seed(SEED)

        train_encodings = tokenization_function(
            X_train_raw
        )

        test_encodings = tokenization_function(
            X_test1_raw
        )

        train_dataset = TextClassificationDataset(
            train_encodings,
            y_train_encoded
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BERT_MICRO_BATCH,
            shuffle=True,
            num_workers=BERT_NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda")
        )

        model = create_bert_model()

        model = train_bert_model(
            model,
            train_loader
        )

        test_probabilities = predict_bert(
            model,
            test_encodings
        )

        np.save(
            test_path,
            test_probabilities
        )

        # Salvar modelo final para eventual inferência no Teste 2
        final_model_dir = (
            f"{OUTPUT_DIR}/{name}_final_model"
        )

        model.save_pretrained(final_model_dir)
        tokenizer.save_pretrained(final_model_dir)

        del model
        del train_dataset
        del train_loader
        del train_encodings
        del test_encodings

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    evaluate_predictions(
        y_test1_encoded,
        test_probabilities,
        name
    )

    return (
        oof_probabilities,
        test_probabilities
    )


# ============================================================
# 10. EXECUTAR OS DOIS BERTs
# ============================================================

bert_standard_oof, bert_standard_test = run_bert_component(
    name="bert_standard",
    tokenization_function=tokenize_conventional
)

bert_headtail_oof, bert_headtail_test = run_bert_component(
    name="bert_headtail",
    tokenization_function=tokenize_head_tail
)

print("\nOs dois BERTs foram concluídos.")

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

BERT tokenizer carregado.

INICIANDO: bert_standard

Treinando fold 1/3


... (14 outputs left out)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3 - Loss: 1.0799
Epoch 2/3 - Loss: 1.0154
Epoch 3/3 - Loss: 0.8941

Treinando modelo final com 100% dos dados.


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3 - Loss: 1.0676
Epoch 2/3 - Loss: 1.0011
Epoch 3/3 - Loss: 0.8892

RESULTADOS: bert_headtail
Accuracy: 0.4656
Macro-F1: 0.4656
Macro-Precision: 0.4661
Macro-Recall: 0.4656

Classification Report:
              precision    recall  f1-score   support

          c1       0.50      0.47      0.48       300
        c234       0.40      0.41      0.40       300
          c5       0.50      0.52      0.51       300

    accuracy                           0.47       900
   macro avg       0.47      0.47      0.47       900
weighted avg       0.47      0.47      0.47       900

Confusion Matrix:
[[141  94  65]
 [ 87 122  91]
 [ 55  89 156]]

Os dois BERTs foram concluídos.


#Qwen3 Embedding: extração dos embeddings e classificadores

In [18]:
# @title
# ============================================================
# 1. CONFIGURAÇÕES QWEN
# ============================================================

QWEN_INSTRUCTION = (
    "Represent the meaning of a Brazilian Portuguese public authority "
    "response for classification according to its clarity, completeness, "
    "and comprehensibility."
)

QWEN_MAX_LENGTH = 512

QWEN_06B = "Qwen/Qwen3-Embedding-0.6B"
QWEN_4B = "Qwen/Qwen3-Embedding-4B"


# ============================================================
# 2. POOLING DO ÚLTIMO TOKEN
# ============================================================

def last_token_pool(last_hidden_states):

    # Os tokens são preenchidos à esquerda.
    # Portanto, a última posição corresponde ao último token real.

    return last_hidden_states[:, -1, :]


# ============================================================
# 3. EXTRAÇÃO DOS EMBEDDINGS
# ============================================================

def extract_qwen_embeddings(
    model_name,
    texts,
    batch_size=4,
    max_length=QWEN_MAX_LENGTH
):

    print("\nCarregando:", model_name)

    use_amp = DEVICE.type == "cuda"

    tokenizer_qwen = AutoTokenizer.from_pretrained(
        model_name,
        padding_side="left",
        trust_remote_code=True
    )

    # Garantir padding à esquerda
    tokenizer_qwen.padding_side = "left"

    model_qwen = AutoModel.from_pretrained(
        model_name,
        torch_dtype=(
            torch.float16 if use_amp else torch.float32
        ),
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )

    model_qwen = model_qwen.to(DEVICE)
    model_qwen.eval()

    formatted_texts = [
        f"Instruct: {QWEN_INSTRUCTION}\nQuery: {text}"
        for text in texts
    ]

    all_embeddings = []

    for start in range(0, len(formatted_texts), batch_size):

        batch_texts = formatted_texts[
            start:start + batch_size
        ]

        inputs = tokenizer_qwen(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        with torch.no_grad():

            if use_amp:

                with torch.cuda.amp.autocast():

                    outputs = model_qwen(**inputs)

                    embeddings = last_token_pool(
                        outputs.last_hidden_state
                    )

            else:

                outputs = model_qwen(**inputs)

                embeddings = last_token_pool(
                    outputs.last_hidden_state
                )

            # Normalização L2
            embeddings = F.normalize(
                embeddings.float(),
                p=2,
                dim=1
            )

        all_embeddings.append(
            embeddings.cpu().numpy()
        )

        processed = min(
            start + batch_size,
            len(formatted_texts)
        )

        if processed % (batch_size * 50) == 0 or processed == len(formatted_texts):

            print(
                f"Processados: {processed}/{len(formatted_texts)}"
            )

    embeddings = np.concatenate(
        all_embeddings,
        axis=0
    )

    del model_qwen
    del tokenizer_qwen
    del all_embeddings

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return embeddings


# ============================================================
# 4. CLASSIFICADOR LOGISTIC REGRESSION
# ============================================================

def run_embedding_classifier(
    name,
    train_embeddings,
    test_embeddings
):

    print("\n" + "=" * 65)
    print("CLASSIFICADOR:", name)
    print("=" * 65)

    oof_path = f"{OUTPUT_DIR}/{name}_oof.npy"
    test_path = f"{OUTPUT_DIR}/{name}_test1.npy"

    if os.path.exists(oof_path) and os.path.exists(test_path):

        print("Previsões encontradas. Carregando...")

        return (
            np.load(oof_path),
            np.load(test_path)
        )

    lr = LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=2000,
        random_state=SEED
    )

    # OOF: cada exemplo é previsto por um classificador
    # que não foi treinado com aquele exemplo.
    oof_probabilities = cross_val_predict(
        lr,
        train_embeddings,
        y_train_encoded,
        cv=FOLDS,
        method="predict_proba",
        n_jobs=1
    )

    # Modelo final com todo o treinamento
    lr.fit(
        train_embeddings,
        y_train_encoded
    )

    test_probabilities = lr.predict_proba(
        test_embeddings
    )

    assert np.array_equal(
        lr.classes_,
        np.array([0, 1, 2])
    )

    np.save(oof_path, oof_probabilities)
    np.save(test_path, test_probabilities)

    evaluate_predictions(
        y_test1_encoded,
        test_probabilities,
        name
    )

    del lr
    gc.collect()

    return (
        oof_probabilities,
        test_probabilities
    )


# ============================================================
# 5. QWEN 0.6B
# ============================================================

qwen06_train_path = (
    f"{OUTPUT_DIR}/qwen06_train_embeddings.npy"
)

qwen06_test_path = (
    f"{OUTPUT_DIR}/qwen06_test1_embeddings.npy"
)

if (
    os.path.exists(qwen06_train_path)
    and os.path.exists(qwen06_test_path)
):

    print("Embeddings Qwen 0.6B encontrados.")

    qwen06_train_embeddings = np.load(
        qwen06_train_path
    )

    qwen06_test_embeddings = np.load(
        qwen06_test_path
    )

else:

    qwen06_train_embeddings = extract_qwen_embeddings(
        QWEN_06B,
        X_train_raw,
        batch_size=4
    )

    np.save(
        qwen06_train_path,
        qwen06_train_embeddings
    )

    qwen06_test_embeddings = extract_qwen_embeddings(
        QWEN_06B,
        X_test1_raw,
        batch_size=4
    )

    np.save(
        qwen06_test_path,
        qwen06_test_embeddings
    )


print(
    "Qwen 0.6B — treinamento:",
    qwen06_train_embeddings.shape
)

print(
    "Qwen 0.6B — Teste 1:",
    qwen06_test_embeddings.shape
)


# Classificador
qwen06_oof, qwen06_test = run_embedding_classifier(
    "qwen06_lr",
    qwen06_train_embeddings,
    qwen06_test_embeddings
)


# ============================================================
# 6. QWEN 4B — OPCIONAL
# ============================================================

qwen4_oof = None
qwen4_test = None

qwen4_train_embeddings = None
qwen4_test_embeddings = None

qwen4_train_path = (
    f"{OUTPUT_DIR}/qwen4_train_embeddings.npy"
)

qwen4_test_path = (
    f"{OUTPUT_DIR}/qwen4_test1_embeddings.npy"
)

# Verificar memória livre
if torch.cuda.is_available():

    free_memory_gb = (
        torch.cuda.mem_get_info()[0] / 1024**3
    )

    print(
        f"\nMemória livre antes do Qwen 4B: "
        f"{free_memory_gb:.2f} GB"
    )

else:

    free_memory_gb = 0


# Critério conservador de execução
RUN_QWEN_4B = (
    torch.cuda.is_available()
    and free_memory_gb >= 12
)

if RUN_QWEN_4B:

    try:

        if (
            os.path.exists(qwen4_train_path)
            and os.path.exists(qwen4_test_path)
        ):

            print("Embeddings Qwen 4B encontrados.")

            qwen4_train_embeddings = np.load(
                qwen4_train_path
            )

            qwen4_test_embeddings = np.load(
                qwen4_test_path
            )

        else:

            qwen4_train_embeddings = extract_qwen_embeddings(
                QWEN_4B,
                X_train_raw,
                batch_size=1
            )

            np.save(
                qwen4_train_path,
                qwen4_train_embeddings
            )

            qwen4_test_embeddings = extract_qwen_embeddings(
                QWEN_4B,
                X_test1_raw,
                batch_size=1
            )

            np.save(
                qwen4_test_path,
                qwen4_test_embeddings
            )

        qwen4_oof, qwen4_test = run_embedding_classifier(
            "qwen4_lr",
            qwen4_train_embeddings,
            qwen4_test_embeddings
        )

    except torch.cuda.OutOfMemoryError:

        print(
            "\nQwen 4B interrompido por falta de memória GPU."
        )

        qwen4_oof = None
        qwen4_test = None

        qwen4_train_embeddings = None
        qwen4_test_embeddings = None

        gc.collect()
        torch.cuda.empty_cache()

else:

    print(
        "\nQwen 4B não será executado nesta sessão. "
        "Memória GPU insuficiente para a tentativa conservadora."
    )


print("\nEtapa Qwen concluída.")


Carregando: Qwen/Qwen3-Embedding-0.6B


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

... (12 outputs left out)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Processados: 50/20092
Processados: 100/20092
Processados: 150/20092
Processados: 200/20092
Processados: 250/20092
Processados: 300/20092
Processados: 350/20092
Processados: 400/20092
Processados: 450/20092
Processados: 500/20092
Processados: 550/20092
Processados: 600/20092
Processados: 650/20092
Processados: 700/20092
Processados: 750/20092
Processados: 800/20092
Processados: 850/20092
Processados: 900/20092
Processados: 950/20092
Processados: 1000/20092
Processados: 1050/20092
Processados: 1100/20092
Processados: 1150/20092
Processados: 1200/20092
Processados: 1250/20092
Processados: 1300/20092
Processados: 1350/20092
Processados: 1400/20092
Processados: 1450/20092
Processados: 1500/20092
Processados: 1550/20092
Processados: 1600/20092
Processados: 1650/20092
Processados: 1700/20092
Processados: 1750/20092
Processados: 1800/20092
Processados: 1850/20092
Processados: 1900/20092
Processados: 1950/20092
Processados: 2000/20092
Processados: 2050/20092
Processados: 2100/20092
Processados:

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Processados: 50/900
Processados: 100/900
Processados: 150/900
Processados: 200/900
Processados: 250/900
Processados: 300/900
Processados: 350/900
Processados: 400/900
Processados: 450/900
Processados: 500/900
Processados: 550/900
Processados: 600/900
Processados: 650/900
Processados: 700/900
Processados: 750/900
Processados: 800/900
Processados: 850/900
Processados: 900/900

CLASSIFICADOR: qwen4_lr

RESULTADOS: qwen4_lr
Accuracy: 0.4989
Macro-F1: 0.492
Macro-Precision: 0.4957
Macro-Recall: 0.4989

Classification Report:
              precision    recall  f1-score   support

          c1       0.52      0.55      0.54       300
        c234       0.47      0.34      0.39       300
          c5       0.50      0.61      0.55       300

    accuracy                           0.50       900
   macro avg       0.50      0.50      0.49       900
weighted avg       0.50      0.50      0.49       900

Confusion Matrix:
[[164  60  76]
 [ 89 102 109]
 [ 60  57 183]]

Etapa Qwen concluída.


#Stacking, avaliação comparativa e exportação

In [19]:
# @title
# ============================================================
# 1. ORGANIZAR OS MODELOS-BASE
# ============================================================

oof_components = [
    word_oof,
    char_oof,
    bert_standard_oof,
    bert_headtail_oof,
    qwen06_oof
]

test_components = [
    word_test,
    char_test,
    bert_standard_test,
    bert_headtail_test,
    qwen06_test
]

component_names = [
    "TFIDF_Word",
    "TFIDF_Char",
    "BERT_Standard",
    "BERT_HeadTail",
    "Qwen_0.6B"
]

# Adicionar Qwen 4B se disponível
if qwen4_oof is not None and qwen4_test is not None:

    oof_components.append(qwen4_oof)
    test_components.append(qwen4_test)

    component_names.append("Qwen_4B")


# ============================================================
# 2. VALIDAR DIMENSÕES E PROBABILIDADES
# ============================================================

for name, oof, test in zip(
    component_names,
    oof_components,
    test_components
):

    assert oof.shape == (len(X_train_raw), 3), (
        f"Dimensão OOF incorreta em {name}: {oof.shape}"
    )

    assert test.shape == (len(X_test1_raw), 3), (
        f"Dimensão Teste 1 incorreta em {name}: {test.shape}"
    )

    assert np.isfinite(oof).all(), (
        f"Valores inválidos no OOF de {name}"
    )

    assert np.isfinite(test).all(), (
        f"Valores inválidos no Teste 1 de {name}"
    )

    assert np.allclose(
        oof.sum(axis=1),
        1,
        atol=1e-4
    ), f"Probabilidades OOF inválidas em {name}"

    assert np.allclose(
        test.sum(axis=1),
        1,
        atol=1e-4
    ), f"Probabilidades de teste inválidas em {name}"


# ============================================================
# 3. CONSTRUIR MATRIZES DO STACKING
# ============================================================

X_meta_train = np.concatenate(
    oof_components,
    axis=1
)

X_meta_test = np.concatenate(
    test_components,
    axis=1
)

print("Modelos incluídos:")
print(component_names)

print("\nDimensões:")

print("X_meta_train:", X_meta_train.shape)
print("X_meta_test:", X_meta_test.shape)

print(
    "\nNúmero de atributos:",
    X_meta_train.shape[1]
)


# ============================================================
# 4. TREINAR META-CLASSIFICADOR
# ============================================================

meta_model = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=2000,
    random_state=SEED
)

meta_model.fit(
    X_meta_train,
    y_train_encoded
)

# Salvar meta-modelo
import joblib

joblib.dump(
    meta_model,
    f"{OUTPUT_DIR}/stacking_meta_model.pkl"
)

print("\nMeta-classificador treinado.")


# ============================================================
# 5. PREVISÕES DO TESTE 1
# ============================================================

stacking_test_probabilities = meta_model.predict_proba(
    X_meta_test
)

stacking_predictions = np.argmax(
    stacking_test_probabilities,
    axis=1
)

np.save(
    f"{OUTPUT_DIR}/stacking_test1.npy",
    stacking_test_probabilities
)

# Avaliação final
evaluate_predictions(
    y_test1_encoded,
    stacking_test_probabilities,
    "STACKING ENSEMBLE"
)


# ============================================================
# 6. COMPARAÇÃO DOS MODELOS
# ============================================================

all_test_probabilities = (
    test_components +
    [stacking_test_probabilities]
)

all_model_names = (
    component_names +
    ["STACKING_ENSEMBLE"]
)

results = []

for name, probabilities in zip(
    all_model_names,
    all_test_probabilities
):

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    results.append({

        "Model": name,

        "Accuracy": accuracy_score(
            y_test1_encoded,
            predictions
        ),

        "Macro-F1": f1_score(
            y_test1_encoded,
            predictions,
            average="macro"
        ),

        "Macro-Precision": precision_score(
            y_test1_encoded,
            predictions,
            average="macro",
            zero_division=0
        ),

        "Macro-Recall": recall_score(
            y_test1_encoded,
            predictions,
            average="macro",
            zero_division=0
        )
    })


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Macro-F1",
    ascending=False
).reset_index(drop=True)

print("\nCOMPARAÇÃO DOS MODELOS")

display(
    results_df.style.format({
        "Accuracy": "{:.4f}",
        "Macro-F1": "{:.4f}",
        "Macro-Precision": "{:.4f}",
        "Macro-Recall": "{:.4f}"
    })
)


# ============================================================
# 7. SALVAR RESULTADOS
# ============================================================

final_predictions = [
    ID_TO_LABEL[index]
    for index in stacking_predictions
]

final_results = pd.DataFrame({

    "resp_text": X_test1_raw,

    "true_clarity": y_test1,

    "predicted_clarity": final_predictions,

    "prob_c1": stacking_test_probabilities[:, 0],

    "prob_c234": stacking_test_probabilities[:, 1],

    "prob_c5": stacking_test_probabilities[:, 2]
})


final_results.to_csv(
    f"{OUTPUT_DIR}/stacking_test1_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

results_df.to_csv(
    f"{OUTPUT_DIR}/model_comparison.csv",
    index=False
)

# Salvar metadados do experimento
experiment_metadata = {

    "seed": SEED,

    "n_splits": N_SPLITS,

    "classes": CLASSES,

    "models": component_names,

    "bert_model": BERT_MODEL_NAME,

    "bert_learning_rate": BERT_LR,

    "bert_epochs": BERT_EPOCHS,

    "bert_effective_batch": BERT_EFFECTIVE_BATCH,

    "bert_max_length": BERT_MAX_LENGTH,

    "qwen_instruction": QWEN_INSTRUCTION,

    "qwen_max_length": QWEN_MAX_LENGTH,

    "meta_model": "LogisticRegression",

    "meta_C": 1.0
}

with open(
    f"{OUTPUT_DIR}/experiment_metadata.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experiment_metadata,
        f,
        ensure_ascii=False,
        indent=4
    )


print("\n" + "=" * 65)
print("EXPERIMENTO CONCLUÍDO")
print("=" * 65)

print("Resultados salvos em:", OUTPUT_DIR)

print("\nArquivos:")

for filename in sorted(os.listdir(OUTPUT_DIR)):

    print("-", filename)

Modelos incluídos:
['TFIDF_Word', 'TFIDF_Char', 'BERT_Standard', 'BERT_HeadTail', 'Qwen_0.6B', 'Qwen_4B']

Dimensões:
X_meta_train: (20092, 18)
X_meta_test: (900, 18)

Número de atributos: 18

Meta-classificador treinado.

RESULTADOS: STACKING ENSEMBLE
Accuracy: 0.5089
Macro-F1: 0.5066
Macro-Precision: 0.5062
Macro-Recall: 0.5089

Classification Report:
              precision    recall  f1-score   support

          c1       0.51      0.55      0.53       300
        c234       0.46      0.40      0.43       300
          c5       0.55      0.57      0.56       300

    accuracy                           0.51       900
   macro avg       0.51      0.51      0.51       900
weighted avg       0.51      0.51      0.51       900

Confusion Matrix:
[[165  78  57]
 [ 93 121  86]
 [ 64  64 172]]

COMPARAÇÃO DOS MODELOS


,Model,Accuracy,Macro-F1,Macro-Precision,Macro-Recall
0,STACKING_ENSEMBLE,0.5089,0.5066,0.5062,0.5089
1,TFIDF_Word,0.4944,0.4938,0.4939,0.4944
2,Qwen_4B,0.4989,0.4920,0.4957,0.4989
3,Qwen_0.6B,0.4844,0.4794,0.4806,0.4844
4,TFIDF_Char,0.4678,0.4661,0.4652,0.4678
5,BERT_HeadTail,0.4656,0.4656,0.4661,0.4656
6,BERT_Standard,0.4600,0.4600,0.4601,0.4600



EXPERIMENTO CONCLUÍDO
Resultados salvos em: /content/ensemble_outputs

Arquivos:
- bert_headtail_final_model
- bert_headtail_fold_1.npy
- bert_headtail_fold_2.npy
- bert_headtail_fold_3.npy
- bert_headtail_oof.npy
- bert_headtail_test1.npy
- bert_standard_final_model
- bert_standard_fold_1.npy
- bert_standard_fold_2.npy
- bert_standard_fold_3.npy
- bert_standard_oof.npy
- bert_standard_test1.npy
- experiment_metadata.json
- model_comparison.csv
- qwen06_lr_oof.npy
- qwen06_lr_test1.npy
- qwen06_test1_embeddings.npy
- qwen06_train_embeddings.npy
- qwen4_lr_oof.npy
- qwen4_lr_test1.npy
- qwen4_test1_embeddings.npy
- qwen4_train_embeddings.npy
- stacking_meta_model.pkl
- stacking_test1.npy
- stacking_test1_predictions.csv
- tfidf_char_oof.npy
- tfidf_char_test1.npy
- tfidf_word_oof.npy
- tfidf_word_test1.npy


#Analisando combinações

In [20]:
# @title
# ============================================================
# COMPARAÇÃO DE DIFERENTES COMBINAÇÕES NO STACKING
# Meta-classificador: Logistic Regression
# Sem novo treinamento dos modelos-base e sem CV adicional
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# ------------------------------------------------------------
# 1. CONFIGURAÇÕES
# ------------------------------------------------------------

OUTPUT_DIR = "/content/ensemble_outputs"
DATA_DIR = "/content/plnEp2/data"

CLASSES = ["c1", "c234", "c5"]

META_C = 1
META_MAX_ITER = 2000

# ------------------------------------------------------------
# 2. CARREGAMENTO DOS LABELS
# ------------------------------------------------------------

train_df = pd.read_csv(f"{DATA_DIR}/train_limpo.csv")

y_train = train_df["clarity"].astype(str).values

test1_labels_path = (
    f"{DATA_DIR}/test1-rotulado-para-teste-estatistico.xlsx"
)

test1_df = pd.read_excel(test1_labels_path)

y_test1 = test1_df["clarity"].astype(str).values

print(f"Treino: {len(y_train)} exemplos")
print(f"Teste 1: {len(y_test1)} exemplos")

# ------------------------------------------------------------
# 3. CARREGAMENTO DAS PREVISÕES OOF E TESTE 1
# ------------------------------------------------------------

def load_array(filename):
    path = os.path.join(OUTPUT_DIR, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Arquivo não encontrado: {path}"
        )

    return np.load(path)


# Previsões OOF dos modelos
oof = {
    "TFIDF_Word": load_array("tfidf_word_oof.npy"),
    "TFIDF_Char": load_array("tfidf_char_oof.npy"),
    "Qwen_06B": load_array("qwen06_lr_oof.npy"),
    "Qwen_4B": load_array("qwen4_lr_oof.npy"),
    "BERT_Standard": load_array("bert_standard_oof.npy"),
}

# Previsões no Teste 1
test1 = {
    "TFIDF_Word": load_array("tfidf_word_test1.npy"),
    "TFIDF_Char": load_array("tfidf_char_test1.npy"),
    "Qwen_06B": load_array("qwen06_lr_test1.npy"),
    "Qwen_4B": load_array("qwen4_lr_test1.npy"),
    "BERT_Standard": load_array("bert_standard_test1.npy"),
    "BERT_HeadTail": load_array("bert_headtail_test1.npy"),
}

# ------------------------------------------------------------
# 4. RECONSTRUÇÃO DO OOF DO BERT HEAD-TAIL
# ------------------------------------------------------------

headtail_oof_path = os.path.join(
    OUTPUT_DIR, "bert_headtail_oof.npy"
)

if os.path.exists(headtail_oof_path):

    oof["BERT_HeadTail"] = np.load(headtail_oof_path)

else:

    print(
        "\nBERT Head-Tail: reconstruindo OOF "
        "a partir das previsões dos folds..."
    )

    if "FOLDS" not in globals():
        raise RuntimeError(
            "Não encontrei a variável FOLDS. "
            "Execute novamente a célula que cria os folds "
            "ou disponibilize bert_headtail_oof.npy."
        )

    headtail_oof = np.zeros(
        (len(y_train), len(CLASSES)),
        dtype=np.float32
    )

    for fold_idx, (_, val_idx) in enumerate(FOLDS, start=1):

        fold_path = os.path.join(
            OUTPUT_DIR,
            f"bert_headtail_fold_{fold_idx}.npy"
        )

        if not os.path.exists(fold_path):
            raise FileNotFoundError(
                f"Previsões do fold não encontradas: {fold_path}"
            )

        fold_predictions = np.load(fold_path)

        if fold_predictions.shape != (
            len(val_idx), len(CLASSES)
        ):
            raise ValueError(
                f"Dimensão incorreta no fold {fold_idx}: "
                f"{fold_predictions.shape}. "
                f"Esperado: {(len(val_idx), len(CLASSES))}"
            )

        headtail_oof[val_idx] = fold_predictions

    oof["BERT_HeadTail"] = headtail_oof

    np.save(headtail_oof_path, headtail_oof)

    print("OOF do BERT Head-Tail reconstruído e salvo.")

# ------------------------------------------------------------
# 5. VERIFICAÇÃO DOS ARRAYS
# ------------------------------------------------------------

print("\nVerificando dimensões dos modelos:")

for model_name in oof:

    train_predictions = oof[model_name]
    test_predictions = test1[model_name]

    if train_predictions.shape != (
        len(y_train), len(CLASSES)
    ):
        raise ValueError(
            f"{model_name}: dimensão OOF incorreta: "
            f"{train_predictions.shape}"
        )

    if test_predictions.shape != (
        len(y_test1), len(CLASSES)
    ):
        raise ValueError(
            f"{model_name}: dimensão Teste 1 incorreta: "
            f"{test_predictions.shape}"
        )

    if not np.isfinite(train_predictions).all():
        raise ValueError(
            f"{model_name}: OOF contém NaN ou infinito."
        )

    if not np.isfinite(test_predictions).all():
        raise ValueError(
            f"{model_name}: Teste 1 contém NaN ou infinito."
        )

    print(
        f"{model_name:18s} | "
        f"OOF: {train_predictions.shape} | "
        f"Teste 1: {test_predictions.shape}"
    )

# Verificação das classes
if not set(np.unique(y_train)).issubset(set(CLASSES)):
    raise ValueError("Há classes inesperadas no treinamento.")

if not set(np.unique(y_test1)).issubset(set(CLASSES)):
    raise ValueError("Há classes inesperadas no Teste 1.")

# ------------------------------------------------------------
# 6. DEFINIÇÃO DAS COMBINAÇÕES
# ------------------------------------------------------------

combinations = {

    "Word + Qwen 4B": [
        "TFIDF_Word",
        "Qwen_4B"
    ],

    "Word + Qwen 4B + Head-Tail": [
        "TFIDF_Word",
        "Qwen_4B",
        "BERT_HeadTail"
    ],

    "Word + Qwen 4B + Qwen 0.6B": [
        "TFIDF_Word",
        "Qwen_4B",
        "Qwen_06B"
    ],

    "Word + Qwen 4B + Char": [
        "TFIDF_Word",
        "Qwen_4B",
        "TFIDF_Char"
    ],

    "Word + Qwen 4B + BERT Standard": [
        "TFIDF_Word",
        "Qwen_4B",
        "BERT_Standard"
    ],

    "Word + Qwen 4B + Head-Tail + Qwen 0.6B": [
        "TFIDF_Word",
        "Qwen_4B",
        "BERT_HeadTail",
        "Qwen_06B"
    ],

    "Word + Qwen 4B + Head-Tail + Char": [
        "TFIDF_Word",
        "Qwen_4B",
        "BERT_HeadTail",
        "TFIDF_Char"
    ],

    "Word + Qwen 4B + Head-Tail + Standard": [
        "TFIDF_Word",
        "Qwen_4B",
        "BERT_HeadTail",
        "BERT_Standard"
    ],

    "Word + Qwen 4B + Head-Tail + Qwen 0.6B + Char": [
        "TFIDF_Word",
        "Qwen_4B",
        "BERT_HeadTail",
        "Qwen_06B",
        "TFIDF_Char"
    ],

    "Todos os seis modelos": [
        "TFIDF_Word",
        "TFIDF_Char",
        "BERT_Standard",
        "BERT_HeadTail",
        "Qwen_06B",
        "Qwen_4B"
    ]
}

# ------------------------------------------------------------
# 7. TREINAMENTO E AVALIAÇÃO DOS META-CLASSIFICADORES
# ------------------------------------------------------------

results = []
predictions_by_combination = {}

for combination_name, selected_models in combinations.items():

    # Concatena probabilidades OOF
    X_meta_train = np.concatenate(
        [oof[name] for name in selected_models],
        axis=1
    )

    # Concatena probabilidades do Teste 1
    X_meta_test = np.concatenate(
        [test1[name] for name in selected_models],
        axis=1
    )

    # Meta-classificador
    meta_model = LogisticRegression(
        C=META_C,
        class_weight="balanced",
        max_iter=META_MAX_ITER
    )

    meta_model.fit(X_meta_train, y_train)

    predictions = meta_model.predict(X_meta_test)

    predictions_by_combination[
        combination_name
    ] = predictions

    # Métricas
    accuracy = accuracy_score(
        y_test1, predictions
    )

    macro_f1 = f1_score(
        y_test1,
        predictions,
        average="macro",
        zero_division=0
    )

    macro_precision = precision_score(
        y_test1,
        predictions,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_test1,
        predictions,
        average="macro",
        zero_division=0
    )

    results.append({
        "Combination": combination_name,
        "Num_Models": len(selected_models),
        "Models": " + ".join(selected_models),
        "Accuracy": accuracy,
        "Macro-F1": macro_f1,
        "Macro-Precision": macro_precision,
        "Macro-Recall": macro_recall
    })

    print(
        f"{combination_name:48s} "
        f"| Acc: {accuracy:.4f} "
        f"| F1: {macro_f1:.4f}"
    )

# ------------------------------------------------------------
# 8. RESULTADOS ORDENADOS
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Macro-F1",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 100)
print("COMPARAÇÃO DAS COMBINAÇÕES DE STACKING")
print("=" * 100)

display(
    results_df[
        [
            "Combination",
            "Num_Models",
            "Accuracy",
            "Macro-F1",
            "Macro-Precision",
            "Macro-Recall"
        ]
    ].style.format({
        "Accuracy": "{:.4f}",
        "Macro-F1": "{:.4f}",
        "Macro-Precision": "{:.4f}",
        "Macro-Recall": "{:.4f}"
    })
)

# ------------------------------------------------------------
# 9. SALVAMENTO
# ------------------------------------------------------------

results_path = os.path.join(
    OUTPUT_DIR,
    "stacking_combinations_comparison.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

# Salva também as previsões de cada combinação
predictions_df = pd.DataFrame(
    predictions_by_combination
)

predictions_path = os.path.join(
    OUTPUT_DIR,
    "stacking_combinations_predictions.csv"
)

predictions_df.to_csv(
    predictions_path,
    index=False
)

print("\nExperimento concluído!")

print(f"\nTabela: {results_path}")
print(f"Previsões: {predictions_path}")

print("\nMelhor combinação por Macro-F1:")

best = results_df.iloc[0]

print(f"Modelos: {best['Combination']}")
print(f"Accuracy: {best['Accuracy']:.4f}")
print(f"Macro-F1: {best['Macro-F1']:.4f}")


Treino: 20092 exemplos
Teste 1: 900 exemplos

Verificando dimensões dos modelos:
TFIDF_Word         | OOF: (20092, 3) | Teste 1: (900, 3)
TFIDF_Char         | OOF: (20092, 3) | Teste 1: (900, 3)
Qwen_06B           | OOF: (20092, 3) | Teste 1: (900, 3)
Qwen_4B            | OOF: (20092, 3) | Teste 1: (900, 3)
BERT_Standard      | OOF: (20092, 3) | Teste 1: (900, 3)
BERT_HeadTail      | OOF: (20092, 3) | Teste 1: (900, 3)
Word + Qwen 4B                                   | Acc: 0.5067 | F1: 0.5020
Word + Qwen 4B + Head-Tail                       | Acc: 0.5078 | F1: 0.5040
Word + Qwen 4B + Qwen 0.6B                       | Acc: 0.5056 | F1: 0.5006
Word + Qwen 4B + Char                            | Acc: 0.5022 | F1: 0.4970
Word + Qwen 4B + BERT Standard                   | Acc: 0.5089 | F1: 0.5054
Word + Qwen 4B + Head-Tail + Qwen 0.6B           | Acc: 0.5122 | F1: 0.5087
Word + Qwen 4B + Head-Tail + Char                | Acc: 0.5111 | F1: 0.5079
Word + Qwen 4B + Head-Tail + Standard        

,Combination,Num_Models,Accuracy,Macro-F1,Macro-Precision,Macro-Recall
0,Word + Qwen 4B + Head-Tail + Qwen 0.6B + Char,5,0.5133,0.5105,0.5097,0.5133
1,Word + Qwen 4B + Head-Tail + Qwen 0.6B,4,0.5122,0.5087,0.5077,0.5122
2,Word + Qwen 4B + Head-Tail + Char,4,0.5111,0.5079,0.5072,0.5111
3,Todos os seis modelos,6,0.5089,0.5066,0.5062,0.5089
4,Word + Qwen 4B + BERT Standard,3,0.5089,0.5054,0.5055,0.5089
5,Word + Qwen 4B + Head-Tail,3,0.5078,0.5040,0.5032,0.5078
6,Word + Qwen 4B,2,0.5067,0.5020,0.5016,0.5067
7,Word + Qwen 4B + Qwen 0.6B,3,0.5056,0.5006,0.5005,0.5056
8,Word + Qwen 4B + Head-Tail + Standard,4,0.5022,0.4986,0.4985,0.5022
9,Word + Qwen 4B + Char,3,0.5022,0.4970,0.4964,0.5022



Experimento concluído!

Tabela: /content/ensemble_outputs/stacking_combinations_comparison.csv
Previsões: /content/ensemble_outputs/stacking_combinations_predictions.csv

Melhor combinação por Macro-F1:
Modelos: Word + Qwen 4B + Head-Tail + Qwen 0.6B + Char
Accuracy: 0.5133
Macro-F1: 0.5105


#Testando a contribuição marginal do qwen e tail

In [21]:
# @title
# ============================================================
# EXPERIMENTO: CONTRIBUIÇÃO MARGINAL DE QWEN E BERT HEAD-TAIL
# Meta-classificador: Logistic Regression
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

OUTPUT_DIR = "/content/ensemble_outputs"

# ------------------------------------------------------------
# 1. CONFIGURAÇÃO
# ------------------------------------------------------------

META_C = 1
META_MAX_ITER = 2000

# ------------------------------------------------------------
# 2. CARREGAR ARRAYS, CASO NÃO ESTEJAM NA MEMÓRIA
# ------------------------------------------------------------

def load_if_needed(dictionary, key, filename):

    if key not in dictionary:
        path = os.path.join(OUTPUT_DIR, filename)

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        dictionary[key] = np.load(path)

    return dictionary[key]


# Reutiliza os dicionários da célula anterior
# ou carrega os arquivos diretamente.

if "oof" not in globals():
    oof = {}

if "test1" not in globals():
    test1 = {}

# OOF
load_if_needed(
    oof, "TFIDF_Word", "tfidf_word_oof.npy"
)
load_if_needed(
    oof, "Qwen_4B", "qwen4_lr_oof.npy"
)
load_if_needed(
    oof, "Qwen_06B", "qwen06_lr_oof.npy"
)
load_if_needed(
    oof, "BERT_HeadTail", "bert_headtail_oof.npy"
)

# Teste 1
load_if_needed(
    test1, "TFIDF_Word", "tfidf_word_test1.npy"
)
load_if_needed(
    test1, "Qwen_4B", "qwen4_lr_test1.npy"
)
load_if_needed(
    test1, "Qwen_06B", "qwen06_lr_test1.npy"
)
load_if_needed(
    test1, "BERT_HeadTail", "bert_headtail_test1.npy"
)

# ------------------------------------------------------------
# 3. DEFINIR COMBINAÇÕES
# ------------------------------------------------------------

combinations = {

    "Word + Head-Tail": [
        "TFIDF_Word",
        "BERT_HeadTail"
    ],

    "Word + Qwen 4B + Head-Tail": [
        "TFIDF_Word",
        "Qwen_4B",
        "BERT_HeadTail"
    ],

    "Word + Qwen 0.6B + Head-Tail": [
        "TFIDF_Word",
        "Qwen_06B",
        "BERT_HeadTail"
    ],

    "Word + Qwen 4B + Qwen 0.6B + Head-Tail": [
        "TFIDF_Word",
        "Qwen_4B",
        "Qwen_06B",
        "BERT_HeadTail"
    ],

    "Qwen 4B + Qwen 0.6B + Head-Tail": [
        "Qwen_4B",
        "Qwen_06B",
        "BERT_HeadTail"
    ]
}

# ------------------------------------------------------------
# 4. VERIFICAR DIMENSÕES
# ------------------------------------------------------------

for model in ["TFIDF_Word", "Qwen_4B",
              "Qwen_06B", "BERT_HeadTail"]:

    assert oof[model].shape == (len(y_train), 3), (
        f"Dimensão OOF incorreta: {model}"
    )

    assert test1[model].shape == (len(y_test1), 3), (
        f"Dimensão Teste 1 incorreta: {model}"
    )

print("Todos os arrays foram verificados.")

# ------------------------------------------------------------
# 5. TREINAR E AVALIAR OS STACKINGS
# ------------------------------------------------------------

results = []
predictions_by_combination = {}

for combination_name, selected_models in combinations.items():

    X_meta_train = np.concatenate(
        [oof[model] for model in selected_models],
        axis=1
    )

    X_meta_test = np.concatenate(
        [test1[model] for model in selected_models],
        axis=1
    )

    meta_model = LogisticRegression(
        C=META_C,
        class_weight="balanced",
        max_iter=META_MAX_ITER
    )

    meta_model.fit(X_meta_train, y_train)

    predictions = meta_model.predict(X_meta_test)

    predictions_by_combination[
        combination_name
    ] = predictions

    results.append({

        "Combination": combination_name,

        "Num_Models": len(selected_models),

        "Accuracy": accuracy_score(
            y_test1, predictions
        ),

        "Macro-F1": f1_score(
            y_test1,
            predictions,
            average="macro",
            zero_division=0
        ),

        "Macro-Precision": precision_score(
            y_test1,
            predictions,
            average="macro",
            zero_division=0
        ),

        "Macro-Recall": recall_score(
            y_test1,
            predictions,
            average="macro",
            zero_division=0
        )
    })

# ------------------------------------------------------------
# 6. ORGANIZAR RESULTADOS
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Macro-F1",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 90)
print("CONTRIBUIÇÃO MARGINAL DOS MODELOS")
print("=" * 90)

display(
    results_df.style.format({
        "Accuracy": "{:.4f}",
        "Macro-F1": "{:.4f}",
        "Macro-Precision": "{:.4f}",
        "Macro-Recall": "{:.4f}"
    })
)

# ------------------------------------------------------------
# 7. SALVAR RESULTADOS
# ------------------------------------------------------------

results_path = os.path.join(
    OUTPUT_DIR,
    "stacking_marginal_comparison.csv"
)

predictions_path = os.path.join(
    OUTPUT_DIR,
    "stacking_marginal_predictions.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

pd.DataFrame(
    predictions_by_combination
).to_csv(
    predictions_path,
    index=False
)

print("\nExperimento concluído!")
print("Resultados:", results_path)
print("Previsões:", predictions_path)

best = results_df.iloc[0]

print("\nMelhor combinação:")
print(best["Combination"])
print(f"Accuracy: {best['Accuracy']:.4f}")
print(f"Macro-F1: {best['Macro-F1']:.4f}")


Todos os arrays foram verificados.

CONTRIBUIÇÃO MARGINAL DOS MODELOS


,Combination,Num_Models,Accuracy,Macro-F1,Macro-Precision,Macro-Recall
0,Word + Qwen 4B + Qwen 0.6B + Head-Tail,4,0.5122,0.5087,0.5077,0.5122
1,Word + Qwen 4B + Head-Tail,3,0.5078,0.5040,0.5032,0.5078
2,Word + Qwen 0.6B + Head-Tail,3,0.5056,0.5017,0.5008,0.5056
3,Word + Head-Tail,2,0.4989,0.4963,0.4954,0.4989
4,Qwen 4B + Qwen 0.6B + Head-Tail,3,0.4922,0.4865,0.4868,0.4922



Experimento concluído!
Resultados: /content/ensemble_outputs/stacking_marginal_comparison.csv
Previsões: /content/ensemble_outputs/stacking_marginal_predictions.csv

Melhor combinação:
Word + Qwen 4B + Qwen 0.6B + Head-Tail
Accuracy: 0.5122
Macro-F1: 0.5087


#Nested Stacking com 5 folds externos e 4 internos (Validação Cruzada de ensemble) encima da melhor combinação

In [ ]:
# @title
# ============================================================
# NESTED STACKING — VALIDAÇÃO CRUZADA
#
# 5 folds externos
# 4 folds internos
#
# Modelos-base:
# 1. TF-IDF Word + Logistic Regression
# 2. Qwen 0.6B + Logistic Regression
# 3. Qwen 4B + Logistic Regression
# 4. BERTimbau Base Head-Tail ORIGINAL
#
# Metaclassificador:
# Logistic Regression
#
# Não utiliza Teste 1 nem Teste 2.
# Não sobrescreve resultados anteriores.
# ============================================================

import os
import json
import math
import random
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

from tqdm.auto import tqdm


# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

NSCV_SEED = 1024

NSCV_OUTER_FOLDS = 5
NSCV_INNER_FOLDS = 4

NSCV_CLASSES = ["c1", "c234", "c5"]

NSCV_LABEL2ID = {
    label: i for i, label in enumerate(NSCV_CLASSES)
}

NSCV_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert torch.cuda.is_available(), "GPU CUDA não encontrada."

# BERT Head-Tail original
NSCV_BERT_NAME = "neuralmind/bert-base-portuguese-cased"

NSCV_BERT_LR = 3e-5
NSCV_BERT_WEIGHT_DECAY = 0.01
NSCV_BERT_EPOCHS = 3

NSCV_MAX_LENGTH = 256
NSCV_HEAD_TOKENS = 127
NSCV_TAIL_TOKENS = 127

NSCV_BATCH_SIZE = 8
NSCV_GRAD_ACCUM = 2
NSCV_WARMUP_RATIO = 0.10

# Metaclassificador
NSCV_META_PARAMS = {
    "C": 1,
    "class_weight": "balanced",
    "max_iter": 2000
}

# TF-IDF Word
NSCV_TFIDF_PARAMS = {
    "analyzer": "word",
    "sublinear_tf": True,
    "ngram_range": (1, 3)
}

NSCV_WORD_LR_PARAMS = {
    "C": 1,
    "class_weight": "balanced",
    "max_iter": 1000
}

# Qwen
NSCV_QWEN_LR_PARAMS = {
    "C": 1,
    "class_weight": "balanced",
    "max_iter": 2000
}

# Diretório exclusivo para esta execução
NSCV_RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

NSCV_OUTPUT_DIR = (
    f"/content/ensemble_outputs/"
    f"nested_stacking_5x4_{NSCV_RUN_ID}"
)

os.makedirs(NSCV_OUTPUT_DIR, exist_ok=True)

print("Dispositivo:", NSCV_DEVICE)
print("Folds externos:", NSCV_OUTER_FOLDS)
print("Folds internos:", NSCV_INNER_FOLDS)
print("Pasta de resultados:", NSCV_OUTPUT_DIR)


# ============================================================
# 2. FUNÇÕES AUXILIARES
# ============================================================

def nscv_set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ============================================================
# 3. CARREGAR DADOS
# ============================================================

NSCV_TRAIN_RAW_PATH = (
    "/content/plnEp2/data/train.csv"
)

NSCV_TRAIN_CLEAN_PATH = (
    "/content/plnEp2/data/train_limpo.csv"
)

NSCV_QWEN06_PATH = (
    "/content/ensemble_outputs/qwen06_train_embeddings.npy"
)

NSCV_QWEN4_PATH = (
    "/content/ensemble_outputs/qwen4_train_embeddings.npy"
)

NSCV_raw_df = pd.read_csv(NSCV_TRAIN_RAW_PATH)
NSCV_clean_df = pd.read_csv(NSCV_TRAIN_CLEAN_PATH)

# Texto original para BERT
NSCV_texts_raw = (
    NSCV_raw_df["resp_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .tolist()
)

# Texto limpo para TF-IDF
NSCV_texts_clean = (
    NSCV_clean_df["resp_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .tolist()
)

NSCV_y = (
    NSCV_raw_df["clarity"]
    .map(NSCV_LABEL2ID)
    .to_numpy()
)

NSCV_y_clean = (
    NSCV_clean_df["clarity"]
    .map(NSCV_LABEL2ID)
    .to_numpy()
)

# Embeddings Qwen previamente extraídos
NSCV_qwen06_embeddings = np.load(NSCV_QWEN06_PATH)
NSCV_qwen4_embeddings = np.load(NSCV_QWEN4_PATH)

# Verificações de alinhamento
NSCV_N = len(NSCV_y)

assert len(NSCV_texts_raw) == NSCV_N
assert len(NSCV_texts_clean) == NSCV_N
assert len(NSCV_y_clean) == NSCV_N

assert np.array_equal(NSCV_y, NSCV_y_clean), (
    "Os rótulos dos arquivos limpo e original não estão alinhados."
)

assert NSCV_qwen06_embeddings.shape[0] == NSCV_N
assert NSCV_qwen4_embeddings.shape[0] == NSCV_N

assert not pd.isna(NSCV_y).any()

print("\nDados carregados:")
print("Exemplos:", NSCV_N)
print("Qwen 0.6B:", NSCV_qwen06_embeddings.shape)
print("Qwen 4B:", NSCV_qwen4_embeddings.shape)

print("\nDistribuição das classes:")
print(
    pd.Series(NSCV_y)
    .map({i: label for i, label in enumerate(NSCV_CLASSES)})
    .value_counts()
)


# ============================================================
# 4. PREPARAR TOKENIZER E TEXTOS DO BERT
# ============================================================

NSCV_tokenizer = AutoTokenizer.from_pretrained(
    NSCV_BERT_NAME
)


def nscv_encode_headtail(text):

    tokens = NSCV_tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    if len(tokens) > (
        NSCV_HEAD_TOKENS + NSCV_TAIL_TOKENS
    ):

        tokens = (
            tokens[:NSCV_HEAD_TOKENS]
            + tokens[-NSCV_TAIL_TOKENS:]
        )

    encoded = NSCV_tokenizer.prepare_for_model(
        tokens,
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=True
    )

    return encoded["input_ids"]


print("\nTokenizando textos para BERT...")

NSCV_bert_ids = [
    nscv_encode_headtail(text)
    for text in tqdm(NSCV_texts_raw)
]

print("Tokenização concluída.")


# ============================================================
# 5. DATASET E COLLATE DO BERT
# ============================================================

class NSCV_BertDataset(Dataset):

    def __init__(self, indices, labels=None):

        self.indices = np.asarray(indices)
        self.labels = labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):

        original_idx = self.indices[idx]

        item = {
            "input_ids": NSCV_bert_ids[original_idx]
        }

        if self.labels is not None:
            item["labels"] = int(
                self.labels[idx]
            )

        return item


def nscv_collate_fn(batch):

    max_len = max(
        len(item["input_ids"])
        for item in batch
    )

    input_ids = []
    attention_masks = []

    for item in batch:

        ids = item["input_ids"]
        padding = max_len - len(ids)

        input_ids.append(
            ids + [
                NSCV_tokenizer.pad_token_id
            ] * padding
        )

        attention_masks.append(
            [1] * len(ids) + [0] * padding
        )

    result = {
        "input_ids": torch.tensor(
            input_ids,
            dtype=torch.long
        ),

        "attention_mask": torch.tensor(
            attention_masks,
            dtype=torch.long
        )
    }

    if "labels" in batch[0]:

        result["labels"] = torch.tensor(
            [item["labels"] for item in batch],
            dtype=torch.long
        )

    return result


# ============================================================
# 6. TREINAR BERT EM UMA DIVISÃO E GERAR PROBABILIDADES
# ============================================================

def nscv_train_bert(
    train_indices,
    predict_indices,
    seed
):

    nscv_set_seed(seed)

    train_labels = NSCV_y[train_indices]

    dataset = NSCV_BertDataset(
        train_indices,
        train_labels
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    loader = DataLoader(
        dataset,
        batch_size=NSCV_BATCH_SIZE,
        shuffle=True,
        collate_fn=nscv_collate_fn,
        generator=generator
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        NSCV_BERT_NAME,
        num_labels=3,
        id2label={
            i: label
            for i, label in enumerate(NSCV_CLASSES)
        },
        label2id=NSCV_LABEL2ID
    ).to(NSCV_DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=NSCV_BERT_LR,
        weight_decay=NSCV_BERT_WEIGHT_DECAY
    )

    steps_per_epoch = math.ceil(
        len(loader) / NSCV_GRAD_ACCUM
    )

    total_steps = (
        steps_per_epoch * NSCV_BERT_EPOCHS
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(
            total_steps * NSCV_WARMUP_RATIO
        ),
        num_training_steps=total_steps
    )

    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(NSCV_BERT_EPOCHS):

        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0

        for step, batch in enumerate(loader):

            batch = {
                key: value.to(NSCV_DEVICE)
                for key, value in batch.items()
            }

            group_start = (
                step // NSCV_GRAD_ACCUM
            ) * NSCV_GRAD_ACCUM

            group_size = min(
                NSCV_GRAD_ACCUM,
                len(loader) - group_start
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                outputs = model(**batch)

                loss = (
                    outputs.loss / group_size
                )

            scaler.scale(loss).backward()

            running_loss += (
                loss.item() * group_size
            )

            group_end = (
                (step + 1) % NSCV_GRAD_ACCUM == 0
                or (step + 1) == len(loader)
            )

            if group_end:

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                scaler.step(optimizer)
                scaler.update()

                scheduler.step()

                optimizer.zero_grad(set_to_none=True)

        print(
            f"    BERT época {epoch + 1}/"
            f"{NSCV_BERT_EPOCHS} | "
            f"Loss: {running_loss / len(loader):.4f}"
        )

    # Predição
    model.eval()

    predict_dataset = NSCV_BertDataset(
        predict_indices
    )

    predict_loader = DataLoader(
        predict_dataset,
        batch_size=NSCV_BATCH_SIZE,
        shuffle=False,
        collate_fn=nscv_collate_fn
    )

    all_probs = []

    with torch.no_grad():

        for batch in predict_loader:

            batch = {
                key: value.to(NSCV_DEVICE)
                for key, value in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = model(**batch).logits

            probs = torch.softmax(
                logits.float(),
                dim=1
            )

            all_probs.append(
                probs.cpu().numpy()
            )

    predictions = np.concatenate(
        all_probs,
        axis=0
    )

    del model
    del optimizer
    del scheduler
    del scaler

    torch.cuda.empty_cache()

    return predictions


# ============================================================
# 7. MODELOS-BASE: TF-IDF E QWEN
# ============================================================

def nscv_predict_tfidf(
    train_indices,
    predict_indices
):

    vectorizer = TfidfVectorizer(
        **NSCV_TFIDF_PARAMS
    )

    X_train = vectorizer.fit_transform(
        [NSCV_texts_clean[i] for i in train_indices]
    )

    X_predict = vectorizer.transform(
        [NSCV_texts_clean[i] for i in predict_indices]
    )

    model = LogisticRegression(
        **NSCV_WORD_LR_PARAMS
    )

    model.fit(
        X_train,
        NSCV_y[train_indices]
    )

    return model.predict_proba(X_predict)


def nscv_predict_qwen(
    embeddings,
    train_indices,
    predict_indices
):

    model = LogisticRegression(
        **NSCV_QWEN_LR_PARAMS
    )

    model.fit(
        embeddings[train_indices],
        NSCV_y[train_indices]
    )

    return model.predict_proba(
        embeddings[predict_indices]
    )


# ============================================================
# 8. VALIDAÇÃO CRUZADA EXTERNA E INTERNA
# ============================================================

NSCV_outer_cv = StratifiedKFold(
    n_splits=NSCV_OUTER_FOLDS,
    shuffle=True,
    random_state=NSCV_SEED
)

NSCV_oof_predictions = np.full(
    NSCV_N,
    -1,
    dtype=int
)

NSCV_oof_probabilities = np.zeros(
    (NSCV_N, 3),
    dtype=np.float32
)

NSCV_outer_results = []

NSCV_fold_files = []

for outer_fold, (
    outer_train_idx,
    outer_val_idx
) in enumerate(
    NSCV_outer_cv.split(
        np.zeros(NSCV_N),
        NSCV_y
    ),
    start=1
):

    print("\n" + "#" * 85)
    print(
        f"FOLD EXTERNO {outer_fold}/"
        f"{NSCV_OUTER_FOLDS}"
    )
    print("#" * 85)

    # --------------------------------------------------------
    # 8.1. Divisão interna
    # --------------------------------------------------------

    NSCV_inner_cv = StratifiedKFold(
        n_splits=NSCV_INNER_FOLDS,
        shuffle=True,
        random_state=NSCV_SEED + outer_fold
    )

    NSCV_outer_train_y = NSCV_y[
        outer_train_idx
    ]

    NSCV_meta_oof = np.zeros(
        (len(outer_train_idx), 12),
        dtype=np.float32
    )

    # Mapeia índices originais para posições locais
    NSCV_local_positions = {
        original_idx: local_idx
        for local_idx, original_idx
        in enumerate(outer_train_idx)
    }

    NSCV_inner_filled = np.zeros(
        len(outer_train_idx),
        dtype=bool
    )

    # --------------------------------------------------------
    # 8.2. Gerar OOF interno para os quatro modelos
    # --------------------------------------------------------

    for inner_fold, (
        inner_train_local,
        inner_val_local
    ) in enumerate(
        NSCV_inner_cv.split(
            np.zeros(len(outer_train_idx)),
            NSCV_outer_train_y
        ),
        start=1
    ):

        print(
            f"\n  Fold interno "
            f"{inner_fold}/{NSCV_INNER_FOLDS}"
        )

        inner_train_idx = outer_train_idx[
            inner_train_local
        ]

        inner_val_idx = outer_train_idx[
            inner_val_local
        ]

        # TF-IDF Word
        print("  Treinando TF-IDF Word...")

        word_probs = nscv_predict_tfidf(
            inner_train_idx,
            inner_val_idx
        )

        # Qwen 0.6B
        print("  Treinando classificador Qwen 0.6B...")

        qwen06_probs = nscv_predict_qwen(
            NSCV_qwen06_embeddings,
            inner_train_idx,
            inner_val_idx
        )

        # Qwen 4B
        print("  Treinando classificador Qwen 4B...")

        qwen4_probs = nscv_predict_qwen(
            NSCV_qwen4_embeddings,
            inner_train_idx,
            inner_val_idx
        )

        # BERT Head-Tail original
        print("  Treinando BERT Head-Tail original...")

        bert_probs = nscv_train_bert(
            inner_train_idx,
            inner_val_idx,
            seed=NSCV_SEED + outer_fold * 100 + inner_fold
        )

        # Concatenar as 12 features
        inner_features = np.hstack([
            word_probs,
            qwen06_probs,
            qwen4_probs,
            bert_probs
        ])

        # Converter índices globais para posições locais
        inner_val_positions = np.array([
            NSCV_local_positions[idx]
            for idx in inner_val_idx
        ])

        NSCV_meta_oof[
            inner_val_positions
        ] = inner_features

        NSCV_inner_filled[
            inner_val_positions
        ] = True

        print(
            "  Features internas geradas:",
            inner_features.shape
        )

    assert NSCV_inner_filled.all(), (
        "Existem exemplos sem probabilidades OOF internas."
    )

    # --------------------------------------------------------
    # 8.3. Treinar modelos-base no conjunto externo completo
    #      e predizer o fold externo
    # --------------------------------------------------------

    print("\nTreinando modelos-base no treino externo completo...")

    # TF-IDF
    NSCV_outer_word_probs = nscv_predict_tfidf(
        outer_train_idx,
        outer_val_idx
    )

    # Qwen 0.6B
    NSCV_outer_qwen06_probs = nscv_predict_qwen(
        NSCV_qwen06_embeddings,
        outer_train_idx,
        outer_val_idx
    )

    # Qwen 4B
    NSCV_outer_qwen4_probs = nscv_predict_qwen(
        NSCV_qwen4_embeddings,
        outer_train_idx,
        outer_val_idx
    )

    # BERT Head-Tail original
    NSCV_outer_bert_probs = nscv_train_bert(
        outer_train_idx,
        outer_val_idx,
        seed=NSCV_SEED + outer_fold * 100
    )

    # Matriz externa de 12 features
    NSCV_outer_meta_features = np.hstack([
        NSCV_outer_word_probs,
        NSCV_outer_qwen06_probs,
        NSCV_outer_qwen4_probs,
        NSCV_outer_bert_probs
    ])

    # --------------------------------------------------------
    # 8.4. Treinar metaclassificador
    # --------------------------------------------------------

    print("\nTreinando metaclassificador...")

    NSCV_meta_model = LogisticRegression(
        **NSCV_META_PARAMS
    )

    NSCV_meta_model.fit(
        NSCV_meta_oof,
        NSCV_outer_train_y
    )

    # Predição no fold externo
    NSCV_fold_probs = NSCV_meta_model.predict_proba(
        NSCV_outer_meta_features
    )

    NSCV_fold_predictions = NSCV_meta_model.predict(
        NSCV_outer_meta_features
    )

    NSCV_oof_probabilities[
        outer_val_idx
    ] = NSCV_fold_probs

    NSCV_oof_predictions[
        outer_val_idx
    ] = NSCV_fold_predictions

    # --------------------------------------------------------
    # 8.5. Métricas do fold externo
    # --------------------------------------------------------

    fold_accuracy = accuracy_score(
        NSCV_y[outer_val_idx],
        NSCV_fold_predictions
    )

    fold_f1 = f1_score(
        NSCV_y[outer_val_idx],
        NSCV_fold_predictions,
        average="macro"
    )

    fold_precision = precision_score(
        NSCV_y[outer_val_idx],
        NSCV_fold_predictions,
        average="macro",
        zero_division=0
    )

    fold_recall = recall_score(
        NSCV_y[outer_val_idx],
        NSCV_fold_predictions,
        average="macro",
        zero_division=0
    )

    NSCV_outer_results.append({
        "Fold": outer_fold,
        "Accuracy": fold_accuracy,
        "Macro-F1": fold_f1,
        "Macro-Precision": fold_precision,
        "Macro-Recall": fold_recall
    })

    print("\nResultados do fold externo:")
    print(f"Accuracy:        {fold_accuracy:.4f}")
    print(f"Macro-F1:        {fold_f1:.4f}")
    print(f"Macro-Precision: {fold_precision:.4f}")
    print(f"Macro-Recall:    {fold_recall:.4f}")

    # --------------------------------------------------------
    # 8.6. Salvar resultados intermediários
    # --------------------------------------------------------

    fold_path = os.path.join(
        NSCV_OUTPUT_DIR,
        f"outer_fold_{outer_fold}_probabilities.npy"
    )

    np.save(
        fold_path,
        NSCV_fold_probs
    )

    NSCV_fold_files.append(fold_path)

    # Salvar progresso
    pd.DataFrame(
        NSCV_outer_results
    ).to_csv(
        os.path.join(
            NSCV_OUTPUT_DIR,
            "outer_fold_progress.csv"
        ),
        index=False
    )


# ============================================================
# 9. RESULTADOS FINAIS
# ============================================================

assert np.all(NSCV_oof_predictions >= 0)

NSCV_results_df = pd.DataFrame(
    NSCV_outer_results
)

print("\n" + "=" * 85)
print("RESULTADOS POR FOLD EXTERNO")
print("=" * 85)

print(
    NSCV_results_df.to_string(
        index=False,
        formatters={
            "Accuracy": "{:.4f}".format,
            "Macro-F1": "{:.4f}".format,
            "Macro-Precision": "{:.4f}".format,
            "Macro-Recall": "{:.4f}".format
        }
    )
)


# ============================================================
# 10. MÉDIA E DESVIO-PADRÃO
# ============================================================

NSCV_summary = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro-F1",
        "Macro-Precision",
        "Macro-Recall"
    ],

    "Mean": [
        NSCV_results_df["Accuracy"].mean(),
        NSCV_results_df["Macro-F1"].mean(),
        NSCV_results_df["Macro-Precision"].mean(),
        NSCV_results_df["Macro-Recall"].mean()
    ],

    "Std": [
        NSCV_results_df["Accuracy"].std(ddof=1),
        NSCV_results_df["Macro-F1"].std(ddof=1),
        NSCV_results_df["Macro-Precision"].std(ddof=1),
        NSCV_results_df["Macro-Recall"].std(ddof=1)
    ]
})

print("\n" + "=" * 85)
print("MÉDIA E DESVIO-PADRÃO — VALIDAÇÃO CRUZADA")
print("=" * 85)

for _, row in NSCV_summary.iterrows():

    print(
        f"{row['Metric']:20s}: "
        f"{row['Mean']:.4f} ± {row['Std']:.4f}"
    )


# ============================================================
# 11. MÉTRICAS AGREGADAS SOBRE TODAS AS PREDIÇÕES OOF
# ============================================================

NSCV_pooled_accuracy = accuracy_score(
    NSCV_y,
    NSCV_oof_predictions
)

NSCV_pooled_f1 = f1_score(
    NSCV_y,
    NSCV_oof_predictions,
    average="macro"
)

NSCV_pooled_precision = precision_score(
    NSCV_y,
    NSCV_oof_predictions,
    average="macro",
    zero_division=0
)

NSCV_pooled_recall = recall_score(
    NSCV_y,
    NSCV_oof_predictions,
    average="macro",
    zero_division=0
)

print("\n" + "=" * 85)
print("MÉTRICAS AGREGADAS — PREDIÇÕES OOF EXTERNAS")
print("=" * 85)

print(f"Accuracy:        {NSCV_pooled_accuracy:.4f}")
print(f"Macro-F1:        {NSCV_pooled_f1:.4f}")
print(f"Macro-Precision: {NSCV_pooled_precision:.4f}")
print(f"Macro-Recall:    {NSCV_pooled_recall:.4f}")


# ============================================================
# 12. RELATÓRIO DE CLASSIFICAÇÃO
# ============================================================

print("\nRELATÓRIO DE CLASSIFICAÇÃO")
print("=" * 85)

print(
    classification_report(
        NSCV_y,
        NSCV_oof_predictions,
        labels=[0, 1, 2],
        target_names=NSCV_CLASSES,
        digits=4,
        zero_division=0
    )
)

print("\nMATRIZ DE CONFUSÃO:")
print(
    confusion_matrix(
        NSCV_y,
        NSCV_oof_predictions,
        labels=[0, 1, 2]
    )
)


# ============================================================
# 13. SALVAR RESULTADOS FINAIS
# ============================================================

NSCV_results_df.to_csv(
    os.path.join(
        NSCV_OUTPUT_DIR,
        "nested_stacking_fold_results.csv"
    ),
    index=False
)

NSCV_summary.to_csv(
    os.path.join(
        NSCV_OUTPUT_DIR,
        "nested_stacking_summary.csv"
    ),
    index=False
)

np.save(
    os.path.join(
        NSCV_OUTPUT_DIR,
        "nested_stacking_oof_predictions.npy"
    ),
    NSCV_oof_predictions
)

np.save(
    os.path.join(
        NSCV_OUTPUT_DIR,
        "nested_stacking_oof_probabilities.npy"
    ),
    NSCV_oof_probabilities
)

NSCV_metadata = {
    "method": "Nested Stacking",
    "outer_folds": NSCV_OUTER_FOLDS,
    "inner_folds": NSCV_INNER_FOLDS,
    "seed": NSCV_SEED,
    "base_models": [
        "TF-IDF Word + Logistic Regression",
        "Qwen 0.6B embeddings + Logistic Regression",
        "Qwen 4B embeddings + Logistic Regression",
        "BERTimbau Base Head-Tail original"
    ],
    "bert_parameters": {
        "learning_rate": NSCV_BERT_LR,
        "weight_decay": NSCV_BERT_WEIGHT_DECAY,
        "epochs": NSCV_BERT_EPOCHS,
        "max_length": NSCV_MAX_LENGTH,
        "head_tokens": NSCV_HEAD_TOKENS,
        "tail_tokens": NSCV_TAIL_TOKENS,
        "batch_size": NSCV_BATCH_SIZE,
        "gradient_accumulation": NSCV_GRAD_ACCUM
    },
    "meta_classifier": {
        "name": "LogisticRegression",
        "parameters": NSCV_META_PARAMS
    },
    "pooled_metrics": {
        "accuracy": float(NSCV_pooled_accuracy),
        "macro_f1": float(NSCV_pooled_f1),
        "macro_precision": float(NSCV_pooled_precision),
        "macro_recall": float(NSCV_pooled_recall)
    }
}

with open(
    os.path.join(
        NSCV_OUTPUT_DIR,
        "nested_stacking_metadata.json"
    ),
    "w"
) as f:

    json.dump(
        NSCV_metadata,
        f,
        indent=4
    )


print("\n" + "=" * 85)
print("NESTED STACKING CONCLUÍDO")
print("=" * 85)

print("Resultados salvos em:")
print(NSCV_OUTPUT_DIR)

print("\nNenhum resultado anterior foi sobrescrito.")

Dispositivo: cuda
Folds externos: 5
Folds internos: 4
Pasta de resultados: /content/ensemble_outputs/nested_stacking_5x4_20261108_033750

Dados carregados:
Exemplos: 20092
Qwen 0.6B: (20092, 1024)
Qwen 4B: (20092, 2560)

Distribuição das classes:
c5      6892
c234    6853
c1      6347
Name: count, dtype: int64

Tokenizando textos para BERT...


  0%|          | 0/20092 [00:00<?, ?it/s]

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Tokenização concluída.

#####################################################################################
FOLD EXTERNO 1/5
#####################################################################################

  Fold interno 1/4
  Treinando TF-IDF Word...
  Treinando classificador Qwen 0.6B...
  Treinando classificador Qwen 4B...
  Treinando BERT Head-Tail original...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    BERT época 1/3 | Loss: 1.0743
    BERT época 2/3 | Loss: 0.9974
    BERT época 3/3 | Loss: 0.8600
  Features internas geradas: (4019, 12)

  Fold interno 2/4
  Treinando TF-IDF Word...
  Treinando classificador Qwen 0.6B...
  Treinando classificador Qwen 4B...
  Treinando BERT Head-Tail original...


... (44 outputs left out)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    BERT época 1/3 | Loss: 1.0772
    BERT época 2/3 | Loss: 1.0002
    BERT época 3/3 | Loss: 0.8620
  Features internas geradas: (4018, 12)

Treinando modelos-base no treino externo completo...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    BERT época 1/3 | Loss: 1.0697
    BERT época 2/3 | Loss: 0.9958
    BERT época 3/3 | Loss: 0.8587

Treinando metaclassificador...

Resultados do fold externo:
Accuracy:        0.4749
Macro-F1:        0.4701
Macro-Precision: 0.4696
Macro-Recall:    0.4762

RESULTADOS POR FOLD EXTERNO
 Fold Accuracy Macro-F1 Macro-Precision Macro-Recall
    1   0.4747   0.4693          0.4695       0.4762
    2   0.4882   0.4803          0.4801       0.4899
    3   0.4741   0.4662          0.4654       0.4754
    4   0.4863   0.4810          0.4818       0.4883
    5   0.4749   0.4701          0.4696       0.4762

MÉDIA E DESVIO-PADRÃO — VALIDAÇÃO CRUZADA
Accuracy            : 0.4796 ± 0.0070
Macro-F1            : 0.4734 ± 0.0068
Macro-Precision     : 0.4733 ± 0.0072
Macro-Recall        : 0.4812 ± 0.0072

MÉTRICAS AGREGADAS — PREDIÇÕES OOF EXTERNAS
Accuracy:        0.4796
Macro-F1:        0.4735
Macro-Precision: 0.4733
Macro-Recall:    0.4812

RELATÓRIO DE CLASSIFICAÇÃO
              precision    rec

#Testando mais hiperparametros para um melhor bert (tail)
##Grid Search do BERT Head-Tail


In [ ]:
# @title
# ============================================================
# FASE 1 — GRID SEARCH EXPLORATÓRIO DO BERT HEAD-TAIL
#
# Objetivo:
# Selecionar hiperparâmetros utilizando uma divisão 80/20.
#
# NÃO sobrescreve:
# - Modelos anteriores
# - Arquivos OOF existentes
# - Previsões do Teste 1
# - Variável FOLDS
# ============================================================

import os
import json
import math
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

from itertools import product

# ------------------------------------------------------------
# 1. CONFIGURAÇÕES EXCLUSIVAS
# ------------------------------------------------------------

HTGS_OUTPUT_DIR = (
    "/content/ensemble_outputs/bert_headtail_gridsearch"
)

os.makedirs(HTGS_OUTPUT_DIR, exist_ok=True)

HTGS_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

HTGS_SEED = 1024

HTGS_MAX_LENGTH = 256
HTGS_HEAD_TOKENS = 127
HTGS_TAIL_TOKENS = 127

HTGS_BATCH_SIZE = 8
HTGS_GRAD_ACCUMULATION = 2
HTGS_EFFECTIVE_BATCH = (
    HTGS_BATCH_SIZE * HTGS_GRAD_ACCUMULATION
)

HTGS_WARMUP_RATIO = 0.10
HTGS_MAX_GRAD_NORM = 1.0

HTGS_CLASSES = ["c1", "c234", "c5"]

HTGS_LABEL2ID = {
    label: idx
    for idx, label in enumerate(HTGS_CLASSES)
}

HTGS_ID2LABEL = {
    idx: label
    for label, idx in HTGS_LABEL2ID.items()
}

# Grid de hiperparâmetros
HTGS_GRID = list(product(
    [2e-5, 3e-5],
    [3, 4],
    [0.01, 0.05]
))

print(f"Combinações: {len(HTGS_GRID)}")
print(f"Batch efetivo: {HTGS_EFFECTIVE_BATCH}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# ------------------------------------------------------------
# 2. FIXAR SEED
# ------------------------------------------------------------

def htgs_set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ------------------------------------------------------------
# 3. CARREGAR DADOS ORIGINAIS
# ------------------------------------------------------------

HTGS_TRAIN_PATH = "/content/plnEp2/data/train.csv"

htgs_train_df = pd.read_csv(HTGS_TRAIN_PATH)

htgs_texts = (
    htgs_train_df["resp_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .tolist()
)

htgs_labels = (
    htgs_train_df["clarity"]
    .astype(str)
    .tolist()
)

if not set(htgs_labels).issubset(set(HTGS_CLASSES)):
    raise ValueError("Foram encontradas classes inesperadas.")

htgs_y = np.array([
    HTGS_LABEL2ID[label]
    for label in htgs_labels
])

# Divisão estratificada 80/20
htgs_indices = np.arange(len(htgs_texts))

htgs_train_idx, htgs_val_idx = train_test_split(
    htgs_indices,
    test_size=0.20,
    random_state=HTGS_SEED,
    stratify=htgs_y
)

htgs_X_train = [htgs_texts[i] for i in htgs_train_idx]
htgs_X_val = [htgs_texts[i] for i in htgs_val_idx]

htgs_y_train = htgs_y[htgs_train_idx]
htgs_y_val = htgs_y[htgs_val_idx]

print("\nDivisão dos dados:")
print(f"Treinamento: {len(htgs_X_train)}")
print(f"Validação: {len(htgs_X_val)}")

print("\nDistribuição no treinamento:")
print(pd.Series(
    [HTGS_ID2LABEL[x] for x in htgs_y_train]
).value_counts(normalize=True).sort_index())

print("\nDistribuição na validação:")
print(pd.Series(
    [HTGS_ID2LABEL[x] for x in htgs_y_val]
).value_counts(normalize=True).sort_index())

# ------------------------------------------------------------
# 4. TOKENIZER E HEAD-TAIL
# ------------------------------------------------------------

htgs_tokenizer = AutoTokenizer.from_pretrained(
    HTGS_MODEL_NAME
)

def htgs_encode_head_tail(text):

    # Tokenização sem tokens especiais
    token_ids = htgs_tokenizer.encode(
        text,
        add_special_tokens=False
    )

    # Preserva início e final
    if len(token_ids) > HTGS_HEAD_TOKENS + HTGS_TAIL_TOKENS:

        token_ids = (
            token_ids[:HTGS_HEAD_TOKENS]
            + token_ids[-HTGS_TAIL_TOKENS:]
        )

    # Adiciona CLS e SEP conforme o tokenizer
    token_ids = htgs_tokenizer.build_inputs_with_special_tokens(
        token_ids
    )

    if len(token_ids) > HTGS_MAX_LENGTH:
        raise ValueError(
            "Sequência excedeu o limite configurado."
        )

    return torch.tensor(
        token_ids,
        dtype=torch.long
    )


# Tokenizamos uma única vez para reutilizar nas oito combinações.
print("\nTokenizando textos...")

htgs_train_tokens = [
    htgs_encode_head_tail(text)
    for text in htgs_X_train
]

htgs_val_tokens = [
    htgs_encode_head_tail(text)
    for text in htgs_X_val
]

print("Tokenização concluída.")

# ------------------------------------------------------------
# 5. DATASET E COLLATE
# ------------------------------------------------------------

class HTGS_Dataset(Dataset):

    def __init__(self, tokenized_texts, labels):

        self.tokenized_texts = tokenized_texts
        self.labels = torch.tensor(
            labels,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        return (
            self.tokenized_texts[idx],
            self.labels[idx]
        )


def htgs_collate_fn(batch):

    sequences, labels = zip(*batch)

    input_ids = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=htgs_tokenizer.pad_token_id
    )

    attention_mask = (
        input_ids != htgs_tokenizer.pad_token_id
    ).long()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": torch.stack(labels)
    }


htgs_train_dataset = HTGS_Dataset(
    htgs_train_tokens,
    htgs_y_train
)

htgs_val_dataset = HTGS_Dataset(
    htgs_val_tokens,
    htgs_y_val
)

# ------------------------------------------------------------
# 6. FUNÇÃO DE AVALIAÇÃO
# ------------------------------------------------------------

def htgs_evaluate(model, dataloader):

    model.eval()

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to("cuda")
            attention_mask = batch["attention_mask"].to("cuda")
            labels = batch["labels"].to("cuda")

            with torch.cuda.amp.autocast():

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )

            predictions = outputs.logits.argmax(dim=1)

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    return macro_f1, accuracy


# ------------------------------------------------------------
# 7. TREINAMENTO DE UMA CONFIGURAÇÃO
# ------------------------------------------------------------

def htgs_train_configuration(
    learning_rate,
    epochs,
    weight_decay,
    configuration_number
):

    htgs_set_seed(HTGS_SEED)

    train_generator = torch.Generator()
    train_generator.manual_seed(HTGS_SEED)

    train_loader = DataLoader(
        htgs_train_dataset,
        batch_size=HTGS_BATCH_SIZE,
        shuffle=True,
        collate_fn=htgs_collate_fn,
        generator=train_generator
    )

    val_loader = DataLoader(
        htgs_val_dataset,
        batch_size=HTGS_BATCH_SIZE,
        shuffle=False,
        collate_fn=htgs_collate_fn
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        HTGS_MODEL_NAME,
        num_labels=len(HTGS_CLASSES),
        id2label=HTGS_ID2LABEL,
        label2id=HTGS_LABEL2ID
    ).cuda()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    updates_per_epoch = math.ceil(
        len(train_loader) / HTGS_GRAD_ACCUMULATION
    )

    total_steps = updates_per_epoch * epochs

    warmup_steps = int(
        total_steps * HTGS_WARMUP_RATIO
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    scaler = torch.cuda.amp.GradScaler()

    best_f1 = -1
    best_accuracy = 0
    best_epoch = 0

    history = []

    print("\n" + "=" * 75)
    print(f"CONFIGURAÇÃO {configuration_number}/{len(HTGS_GRID)}")
    print(f"LR: {learning_rate}")
    print(f"Epochs: {epochs}")
    print(f"Weight decay: {weight_decay}")
    print("=" * 75)

    for epoch in range(epochs):

        model.train()

        optimizer.zero_grad(set_to_none=True)

        epoch_loss = 0

        for step, batch in enumerate(train_loader):

            input_ids = batch["input_ids"].cuda()
            attention_mask = batch["attention_mask"].cuda()
            labels = batch["labels"].cuda()

            # Corrige a escala da perda inclusive no último
            # grupo de acumulação, caso seja incompleto.
            group_start = (
                step // HTGS_GRAD_ACCUMULATION
            ) * HTGS_GRAD_ACCUMULATION

            group_size = min(
                HTGS_GRAD_ACCUMULATION,
                len(train_loader) - group_start
            )

            with torch.cuda.amp.autocast():

                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                loss = outputs.loss / group_size

            scaler.scale(loss).backward()

            epoch_loss += loss.item() * group_size

            is_accumulation_step = (
                (step + 1) % HTGS_GRAD_ACCUMULATION == 0
            )

            is_last_batch = (
                step + 1 == len(train_loader)
            )

            if is_accumulation_step or is_last_batch:

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    HTGS_MAX_GRAD_NORM
                )

                scaler.step(optimizer)
                scaler.update()

                scheduler.step()

                optimizer.zero_grad(set_to_none=True)

        val_f1, val_accuracy = htgs_evaluate(
            model,
            val_loader
        )

        mean_loss = epoch_loss / len(train_loader)

        history.append({
            "epoch": epoch + 1,
            "train_loss": mean_loss,
            "val_macro_f1": val_f1,
            "val_accuracy": val_accuracy
        })

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Loss: {mean_loss:.4f} | "
            f"Val Macro-F1: {val_f1:.4f} | "
            f"Val Accuracy: {val_accuracy:.4f}"
        )

        if val_f1 > best_f1:

            best_f1 = val_f1
            best_accuracy = val_accuracy
            best_epoch = epoch + 1

    del model
    del optimizer
    del scheduler
    del scaler

    torch.cuda.empty_cache()

    return {
        "learning_rate": learning_rate,
        "epochs": epochs,
        "weight_decay": weight_decay,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_f1,
        "best_val_accuracy": best_accuracy,
        "history": history
    }


# ------------------------------------------------------------
# 8. EXECUTAR GRID SEARCH
# ------------------------------------------------------------

htgs_results = []

for config_idx, (
    learning_rate,
    epochs,
    weight_decay
) in enumerate(HTGS_GRID, start=1):

    result = htgs_train_configuration(
        learning_rate=learning_rate,
        epochs=epochs,
        weight_decay=weight_decay,
        configuration_number=config_idx
    )

    htgs_results.append(result)

    # Salva progresso a cada configuração
    htgs_progress = pd.DataFrame([
        {
            "Learning_Rate": r["learning_rate"],
            "Epochs": r["epochs"],
            "Weight_Decay": r["weight_decay"],
            "Best_Epoch": r["best_epoch"],
            "Macro_F1": r["best_val_macro_f1"],
            "Accuracy": r["best_val_accuracy"]
        }
        for r in htgs_results
    ])

    htgs_progress.to_csv(
        os.path.join(
            HTGS_OUTPUT_DIR,
            "gridsearch_progress.csv"
        ),
        index=False
    )

# ------------------------------------------------------------
# 9. RESULTADOS FINAIS
# ------------------------------------------------------------

htgs_results_df = pd.DataFrame([
    {
        "Learning_Rate": r["learning_rate"],
        "Epochs": r["epochs"],
        "Weight_Decay": r["weight_decay"],
        "Best_Epoch": r["best_epoch"],
        "Macro_F1": r["best_val_macro_f1"],
        "Accuracy": r["best_val_accuracy"]
    }
    for r in htgs_results
])

htgs_results_df = htgs_results_df.sort_values(
    "Macro_F1",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 90)
print("RESULTADOS DO GRID SEARCH — BERT HEAD-TAIL")
print("=" * 90)

display(
    htgs_results_df.style.format({
        "Learning_Rate": "{:.1e}",
        "Weight_Decay": "{:.3f}",
        "Macro_F1": "{:.4f}",
        "Accuracy": "{:.4f}"
    })
)

# ------------------------------------------------------------
# 10. SALVAR RESULTADOS E MELHOR CONFIGURAÇÃO
# ------------------------------------------------------------

htgs_results_path = os.path.join(
    HTGS_OUTPUT_DIR,
    "gridsearch_results.csv"
)

htgs_metadata_path = os.path.join(
    HTGS_OUTPUT_DIR,
    "best_configuration.json"
)

htgs_results_df.to_csv(
    htgs_results_path,
    index=False
)

htgs_best = htgs_results_df.iloc[0]

htgs_best_config = {
    "model": HTGS_MODEL_NAME,
    "learning_rate": float(htgs_best["Learning_Rate"]),
    "epochs": int(htgs_best["Epochs"]),
    "weight_decay": float(htgs_best["Weight_Decay"]),
    "best_epoch": int(htgs_best["Best_Epoch"]),
    "validation_macro_f1": float(htgs_best["Macro_F1"]),
    "validation_accuracy": float(htgs_best["Accuracy"]),
    "max_length": HTGS_MAX_LENGTH,
    "head_tokens": HTGS_HEAD_TOKENS,
    "tail_tokens": HTGS_TAIL_TOKENS,
    "batch_size": HTGS_BATCH_SIZE,
    "gradient_accumulation": HTGS_GRAD_ACCUMULATION,
    "warmup_ratio": HTGS_WARMUP_RATIO,
    "seed": HTGS_SEED
}

with open(htgs_metadata_path, "w") as f:
    json.dump(
        htgs_best_config,
        f,
        indent=4
    )

print("\nMelhor configuração encontrada:")
print(json.dumps(htgs_best_config, indent=4))

print("\nArquivos salvos em:")
print(HTGS_OUTPUT_DIR)

Combinações: 8
Batch efetivo: 16
GPU: NVIDIA A100-SXM4-40GB

Divisão dos dados:
Treinamento: 16073
Validação: 4019

Distribuição no treinamento:
c1      0.315934
c234    0.341069
c5      0.342998
Name: proportion, dtype: float64

Distribuição na validação:
c1      0.31575
c234    0.34113
c5      0.34312
Name: proportion, dtype: float64

Tokenizando textos...
Tokenização concluída.


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



CONFIGURAÇÃO 1/8
LR: 2e-05
Epochs: 3
Weight decay: 0.01
Epoch 1/3 | Loss: 1.0676 | Val Macro-F1: 0.4431 | Val Accuracy: 0.4451
Epoch 2/3 | Loss: 0.9937 | Val Macro-F1: 0.4525 | Val Accuracy: 0.4509
Epoch 3/3 | Loss: 0.8789 | Val Macro-F1: 0.4571 | Val Accuracy: 0.4613


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



CONFIGURAÇÃO 2/8
LR: 2e-05
Epochs: 3
Weight decay: 0.05
Epoch 1/3 | Loss: 1.0699 | Val Macro-F1: 0.4443 | Val Accuracy: 0.4444
Epoch 2/3 | Loss: 1.0016 | Val Macro-F1: 0.4498 | Val Accuracy: 0.4491
Epoch 3/3 | Loss: 0.8972 | Val Macro-F1: 0.4580 | Val Accuracy: 0.4608


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


... (9 outputs left out)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



CONFIGURAÇÃO 8/8
LR: 3e-05
Epochs: 4
Weight decay: 0.05
Epoch 1/4 | Loss: 1.0713 | Val Macro-F1: 0.4540 | Val Accuracy: 0.4538
Epoch 2/4 | Loss: 1.0025 | Val Macro-F1: 0.4382 | Val Accuracy: 0.4399
Epoch 3/4 | Loss: 0.8602 | Val Macro-F1: 0.4560 | Val Accuracy: 0.4603
Epoch 4/4 | Loss: 0.6815 | Val Macro-F1: 0.4557 | Val Accuracy: 0.4563

RESULTADOS DO GRID SEARCH — BERT HEAD-TAIL


,Learning_Rate,Epochs,Weight_Decay,Best_Epoch,Macro_F1,Accuracy
0,3.0e-05,3,0.050,3,0.4607,0.4616
1,2.0e-05,3,0.050,3,0.4580,0.4608
2,2.0e-05,4,0.050,3,0.4579,0.4618
3,2.0e-05,4,0.010,3,0.4576,0.4593
4,3.0e-05,3,0.010,3,0.4573,0.4601
5,2.0e-05,3,0.010,3,0.4571,0.4613
6,3.0e-05,4,0.050,3,0.4560,0.4603
7,3.0e-05,4,0.010,3,0.4491,0.4524



Melhor configuração encontrada:
{
    "model": "neuralmind/bert-base-portuguese-cased",
    "learning_rate": 3e-05,
    "epochs": 3,
    "weight_decay": 0.05,
    "best_epoch": 3,
    "validation_macro_f1": 0.46066018036078643,
    "validation_accuracy": 0.4615576013933814,
    "max_length": 256,
    "head_tokens": 127,
    "tail_tokens": 127,
    "batch_size": 8,
    "gradient_accumulation": 2,
    "warmup_ratio": 0.1,
    "seed": 1024
}

Arquivos salvos em:
/content/ensemble_outputs/bert_headtail_gridsearch


In [ ]:
# @title
# ============================================================
# FASE 2 — BERT HEAD-TAIL OTIMIZADO
# Geração de probabilidades OOF e Teste 1
#
# Não sobrescreve os resultados anteriores.
# Prefixo exclusivo: HTGA_
# ============================================================

import os
import json
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. Configuração e verificações
# ------------------------------------------------------------

HTGA_OUTPUT_DIR = "/content/ensemble_outputs/bert_headtail_optimized"
os.makedirs(HTGA_OUTPUT_DIR, exist_ok=True)

HTGA_MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
HTGA_CLASSES = ["c1", "c234", "c5"]
HTGA_LABEL2ID = {label: i for i, label in enumerate(HTGA_CLASSES)}
HTGA_ID2LABEL = {i: label for label, i in HTGA_LABEL2ID.items()}

HTGA_SEED = 1024
HTGA_MAX_LENGTH = 256
HTGA_HEAD_TOKENS = 127
HTGA_TAIL_TOKENS = 127
HTGA_BATCH_SIZE = 8
HTGA_GRAD_ACCUM = 2
HTGA_WARMUP_RATIO = 0.10

HTGA_LR = 3e-5
HTGA_WEIGHT_DECAY = 0.05
HTGA_EPOCHS = 3

HTGA_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert torch.cuda.is_available(), "GPU CUDA não está disponível."
assert "FOLDS" in globals(), "A variável FOLDS não foi encontrada."

print("Dispositivo:", HTGA_DEVICE)
print("Número de folds:", len(FOLDS))
print("Configuração:", {
    "learning_rate": HTGA_LR,
    "weight_decay": HTGA_WEIGHT_DECAY,
    "epochs": HTGA_EPOCHS
})

# ------------------------------------------------------------
# 2. Carregar os dados originais
# ------------------------------------------------------------

HTGA_TRAIN_PATH = "/content/plnEp2/data/train.csv"
HTGA_TEST1_PATH = "/content/plnEp2/data/test1.xlsx"

HTGA_train_df = pd.read_csv(HTGA_TRAIN_PATH)
HTGA_test1_df = pd.read_excel(HTGA_TEST1_PATH)

HTGA_X = (
    HTGA_train_df["resp_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .tolist()
)

HTGA_y = (
    HTGA_train_df["clarity"]
    .map(HTGA_LABEL2ID)
    .to_numpy()
)

HTGA_X_test1 = (
    HTGA_test1_df["resp_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .tolist()
)

assert not pd.isna(HTGA_y).any(), "Há rótulos não reconhecidos no treino."
assert len(HTGA_X) == len(HTGA_y)
assert len(FOLDS) >= 2

print("Treino:", len(HTGA_X))
print("Teste 1:", len(HTGA_X_test1))

# ------------------------------------------------------------
# 3. Seeds
# ------------------------------------------------------------

def htga_set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

htga_set_seed(HTGA_SEED)

# ------------------------------------------------------------
# 4. Tokenizer e estratégia Head-Tail
# ------------------------------------------------------------

HTGA_tokenizer = AutoTokenizer.from_pretrained(HTGA_MODEL_NAME)

def htga_encode_text(text):
    tokens = HTGA_tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    if len(tokens) > HTGA_HEAD_TOKENS + HTGA_TAIL_TOKENS:
        tokens = (
            tokens[:HTGA_HEAD_TOKENS]
            + tokens[-HTGA_TAIL_TOKENS:]
        )

    encoded = HTGA_tokenizer.prepare_for_model(
        tokens,
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=True
    )

    return encoded["input_ids"]

print("Tokenizando treinamento e Teste 1...")

HTGA_train_ids = [
    htga_encode_text(text) for text in tqdm(HTGA_X)
]

HTGA_test1_ids = [
    htga_encode_text(text) for text in tqdm(HTGA_X_test1)
]

# ------------------------------------------------------------
# 5. Dataset e collate
# ------------------------------------------------------------

class HTGA_Dataset(Dataset):
    def __init__(self, input_ids, labels=None):
        self.input_ids = input_ids
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        item = {"input_ids": self.input_ids[idx]}

        if self.labels is not None:
            item["labels"] = int(self.labels[idx])

        return item


def htga_collate_fn(batch):
    max_len = max(len(item["input_ids"]) for item in batch)

    input_ids = []
    attention_masks = []

    for item in batch:
        ids = item["input_ids"]
        padding = max_len - len(ids)

        input_ids.append(
            ids + [HTGA_tokenizer.pad_token_id] * padding
        )

        attention_masks.append(
            [1] * len(ids) + [0] * padding
        )

    result = {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long)
    }

    if "labels" in batch[0]:
        result["labels"] = torch.tensor(
            [item["labels"] for item in batch],
            dtype=torch.long
        )

    return result


# ------------------------------------------------------------
# 6. Treinamento e predição
# ------------------------------------------------------------

def htga_train_model(train_indices, validation_indices=None,
                     full_train=False):

    htga_set_seed(HTGA_SEED)

    if full_train:
        HTGA_current_train_ids = HTGA_train_ids
        HTGA_current_y = HTGA_y
    else:
        HTGA_current_train_ids = [
            HTGA_train_ids[i] for i in train_indices
        ]
        HTGA_current_y = HTGA_y[train_indices]

    HTGA_train_dataset = HTGA_Dataset(
        HTGA_current_train_ids,
        HTGA_current_y
    )

    HTGA_train_generator = torch.Generator()
    HTGA_train_generator.manual_seed(HTGA_SEED)

    HTGA_train_loader = DataLoader(
        HTGA_train_dataset,
        batch_size=HTGA_BATCH_SIZE,
        shuffle=True,
        collate_fn=htga_collate_fn,
        generator=HTGA_train_generator
    )

    HTGA_model = AutoModelForSequenceClassification.from_pretrained(
        HTGA_MODEL_NAME,
        num_labels=3,
        id2label=HTGA_ID2LABEL,
        label2id=HTGA_LABEL2ID
    ).to(HTGA_DEVICE)

    HTGA_optimizer = torch.optim.AdamW(
        HTGA_model.parameters(),
        lr=HTGA_LR,
        weight_decay=HTGA_WEIGHT_DECAY
    )

    HTGA_total_steps = (
        len(HTGA_train_loader) // HTGA_GRAD_ACCUM
        + int(len(HTGA_train_loader) % HTGA_GRAD_ACCUM != 0)
    ) * HTGA_EPOCHS

    HTGA_scheduler = get_linear_schedule_with_warmup(
        HTGA_optimizer,
        num_warmup_steps=int(
            HTGA_total_steps * HTGA_WARMUP_RATIO
        ),
        num_training_steps=HTGA_total_steps
    )

    HTGA_scaler = torch.amp.GradScaler("cuda")

    for epoch in range(HTGA_EPOCHS):
        HTGA_model.train()
        HTGA_optimizer.zero_grad(set_to_none=True)

        HTGA_running_loss = 0.0

        for step, batch in enumerate(
            tqdm(HTGA_train_loader, desc=f"Época {epoch+1}/{HTGA_EPOCHS}")
        ):
            HTGA_batch = {
                key: value.to(HTGA_DEVICE)
                for key, value in batch.items()
            }

            HTGA_group_start = (
                step // HTGA_GRAD_ACCUM
            ) * HTGA_GRAD_ACCUM

            HTGA_group_size = min(
                HTGA_GRAD_ACCUM,
                len(HTGA_train_loader) - HTGA_group_start
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                HTGA_outputs = HTGA_model(**HTGA_batch)
                HTGA_loss = HTGA_outputs.loss / HTGA_group_size

            HTGA_scaler.scale(HTGA_loss).backward()
            HTGA_running_loss += HTGA_loss.item() * HTGA_group_size

            HTGA_is_group_end = (
                (step + 1) % HTGA_GRAD_ACCUM == 0
                or (step + 1) == len(HTGA_train_loader)
            )

            if HTGA_is_group_end:
                HTGA_scaler.unscale_(HTGA_optimizer)

                torch.nn.utils.clip_grad_norm_(
                    HTGA_model.parameters(),
                    max_norm=1.0
                )

                HTGA_scaler.step(HTGA_optimizer)
                HTGA_scaler.update()
                HTGA_scheduler.step()
                HTGA_optimizer.zero_grad(set_to_none=True)

        print(
            f"Época {epoch+1}: "
            f"Loss médio = {HTGA_running_loss / len(HTGA_train_loader):.4f}"
        )

    return HTGA_model


def htga_predict_proba(model, input_ids):
    model.eval()

    HTGA_dataset = HTGA_Dataset(input_ids)

    HTGA_loader = DataLoader(
        HTGA_dataset,
        batch_size=HTGA_BATCH_SIZE,
        shuffle=False,
        collate_fn=htga_collate_fn
    )

    HTGA_all_probs = []

    with torch.no_grad():
        for batch in tqdm(HTGA_loader, desc="Gerando probabilidades"):
            HTGA_batch = {
                key: value.to(HTGA_DEVICE)
                for key, value in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                HTGA_logits = model(**HTGA_batch).logits

            HTGA_probs = torch.softmax(
                HTGA_logits.float(),
                dim=1
            )

            HTGA_all_probs.append(
                HTGA_probs.cpu().numpy()
            )

    return np.concatenate(HTGA_all_probs, axis=0)


# ------------------------------------------------------------
# 7. Gerar probabilidades OOF
# ------------------------------------------------------------

HTGA_oof_probs = np.zeros(
    (len(HTGA_X), len(HTGA_CLASSES)),
    dtype=np.float32
)

HTGA_oof_filled = np.zeros(len(HTGA_X), dtype=bool)

for HTGA_fold_number, (HTGA_train_idx, HTGA_val_idx) in enumerate(
    FOLDS, start=1
):

    print("\n" + "=" * 65)
    print(f"FOLD {HTGA_fold_number}/{len(FOLDS)}")
    print("=" * 65)

    HTGA_fold_model = htga_train_model(
        train_indices=HTGA_train_idx,
        validation_indices=HTGA_val_idx,
        full_train=False
    )

    HTGA_fold_probs = htga_predict_proba(
        HTGA_fold_model,
        [HTGA_train_ids[i] for i in HTGA_val_idx]
    )

    HTGA_oof_probs[HTGA_val_idx] = HTGA_fold_probs
    HTGA_oof_filled[HTGA_val_idx] = True

    del HTGA_fold_model
    torch.cuda.empty_cache()

assert HTGA_oof_filled.all(), (
    "Existem exemplos sem probabilidades OOF. "
    "Verifique se FOLDS cobre todo o treinamento."
)

# ------------------------------------------------------------
# 8. Treinar com 100% do treino e prever Teste 1
# ------------------------------------------------------------

print("\nTreinando modelo final com 100% do treinamento...")

HTGA_full_model = htga_train_model(
    train_indices=None,
    full_train=True
)

HTGA_test1_probs = htga_predict_proba(
    HTGA_full_model,
    HTGA_test1_ids
)

# ------------------------------------------------------------
# 9. Salvar tudo em novos arquivos
# ------------------------------------------------------------

HTGA_oof_path = os.path.join(
    HTGA_OUTPUT_DIR,
    "bert_headtail_optimized_oof.npy"
)

HTGA_test1_path = os.path.join(
    HTGA_OUTPUT_DIR,
    "bert_headtail_optimized_test1.npy"
)

HTGA_metadata_path = os.path.join(
    HTGA_OUTPUT_DIR,
    "bert_headtail_optimized_metadata.json"
)

np.save(HTGA_oof_path, HTGA_oof_probs)
np.save(HTGA_test1_path, HTGA_test1_probs)

HTGA_metadata = {
    "model": HTGA_MODEL_NAME,
    "learning_rate": HTGA_LR,
    "weight_decay": HTGA_WEIGHT_DECAY,
    "epochs": HTGA_EPOCHS,
    "best_epoch_from_grid_search": 3,
    "max_length": HTGA_MAX_LENGTH,
    "head_tokens": HTGA_HEAD_TOKENS,
    "tail_tokens": HTGA_TAIL_TOKENS,
    "batch_size": HTGA_BATCH_SIZE,
    "gradient_accumulation": HTGA_GRAD_ACCUM,
    "seed": HTGA_SEED,
    "classes": HTGA_CLASSES,
    "oof_shape": list(HTGA_oof_probs.shape),
    "test1_shape": list(HTGA_test1_probs.shape)
}

with open(HTGA_metadata_path, "w") as f:
    json.dump(HTGA_metadata, f, indent=4)

print("\nArquivos salvos:")
print(HTGA_oof_path)
print(HTGA_test1_path)
print(HTGA_metadata_path)

print("\nDimensões OOF:", HTGA_oof_probs.shape)
print("Dimensões Teste 1:", HTGA_test1_probs.shape)

print("\nFase 2 concluída.")

Dispositivo: cuda
Número de folds: 3
Configuração: {'learning_rate': 3e-05, 'weight_decay': 0.05, 'epochs': 3}
Treino: 20092
Teste 1: 900
Tokenizando treinamento e Teste 1...


  0%|          | 0/20092 [00:00<?, ?it/s]

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


  0%|          | 0/900 [00:00<?, ?it/s]


FOLD 1/3


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


... (31 outputs left out)

Época 3/3:   0%|          | 0/2512 [00:00<?, ?it/s]

Época 3: Loss médio = 0.8719


Gerando probabilidades:   0%|          | 0/113 [00:00<?, ?it/s]


Arquivos salvos:
/content/ensemble_outputs/bert_headtail_optimized/bert_headtail_optimized_oof.npy
/content/ensemble_outputs/bert_headtail_optimized/bert_headtail_optimized_test1.npy
/content/ensemble_outputs/bert_headtail_optimized/bert_headtail_optimized_metadata.json

Dimensões OOF: (20092, 3)
Dimensões Teste 1: (900, 3)

Fase 2 concluída.


In [ ]:
# @title
# ============================================================
# DIAGNÓSTICO — ARQUIVOS DE PROBABILIDADES DOS ENSEMBLES
# Não modifica nem sobrescreve arquivos.
# ============================================================

import os
import numpy as np

DIAG_ROOT = "/content/ensemble_outputs"

print("=" * 75)
print("ARQUIVOS .NPY ENCONTRADOS")
print("=" * 75)

for root, dirs, files in os.walk(DIAG_ROOT):
    for filename in sorted(files):
        if filename.endswith(".npy"):
            path = os.path.join(root, filename)

            try:
                arr = np.load(path, mmap_mode="r")
                print(f"\nArquivo: {path}")
                print(f"Dimensões: {arr.shape}")
                print(f"Tipo: {arr.dtype}")
            except Exception as e:
                print(f"\nErro ao ler {path}: {e}")

print("\n" + "=" * 75)
print("VARIÁVEIS DE PROBABILIDADES DISPONÍVEIS NA MEMÓRIA")
print("=" * 75)

for name, value in list(globals().items()):
    if isinstance(value, np.ndarray) and value.ndim == 2:
        if value.shape[1] == 3:
            print(f"{name}: {value.shape}")

ARQUIVOS .NPY ENCONTRADOS

Arquivo: /content/ensemble_outputs/bert_headtail_fold_1.npy
Dimensões: (6698, 3)
Tipo: float32

Arquivo: /content/ensemble_outputs/bert_headtail_fold_2.npy
Dimensões: (6697, 3)
Tipo: float32

Arquivo: /content/ensemble_outputs/bert_headtail_fold_3.npy
Dimensões: (6697, 3)
Tipo: float32

Arquivo: /content/ensemble_outputs/bert_headtail_oof.npy
Dimensões: (20092, 3)
Tipo: float32

Arquivo: /content/ensemble_outputs/bert_headtail_test1.npy
Dimensões: (900, 3)
Tipo: float32

Arquivo: /content/ensemble_outputs/bert_standard_fold_1.npy
Dimensões: (6698, 3)
Tipo: float32

Arquivo: /content/ensemble_outputs/bert_standard_fold_2.npy
Dimensões: (6697, 3)
Tipo: float32

Arquivo: /content/ensemble_outputs/bert_standard_fold_3.npy
Dimensões: (6697, 3)
Tipo: float32

Arquivo: /content/ensemble_outputs/bert_standard_oof.npy
Dimensões: (20092, 3)
Tipo: float32

Arquivo: /content/ensemble_outputs/bert_standard_test1.npy
Dimensões: (900, 3)
Tipo: float32

Arquivo: /content/ens

#Refazendo a combinação com Reg Log com o modelo bert head-tail com os novos hiperparametros

In [ ]:
# @title
# ============================================================
# FASE 3 — AVALIAÇÃO DO STACKING COM BERT HEAD-TAIL OTIMIZADO
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

HT3_DIR = "/content/ensemble_outputs"
HT3_OUTPUT_DIR = os.path.join(
    HT3_DIR,
    "stacking_headtail_optimized"
)
os.makedirs(HT3_OUTPUT_DIR, exist_ok=True)

HT3_CLASSES = ["c1", "c234", "c5"]
HT3_LABEL2ID = {label: i for i, label in enumerate(HT3_CLASSES)}

# ------------------------------------------------------------
# 1. Carregar probabilidades OOF
# ------------------------------------------------------------

HT3_word_oof = np.load(
    f"{HT3_DIR}/tfidf_word_oof.npy"
)

HT3_qwen06_oof = np.load(
    f"{HT3_DIR}/qwen06_lr_oof.npy"
)

HT3_qwen4_oof = np.load(
    f"{HT3_DIR}/qwen4_lr_oof.npy"
)

HT3_headtail_oof = np.load(
    f"{HT3_DIR}/bert_headtail_optimized/"
    "bert_headtail_optimized_oof.npy"
)

# ------------------------------------------------------------
# 2. Carregar probabilidades do Teste 1
# ------------------------------------------------------------

HT3_word_test = np.load(
    f"{HT3_DIR}/tfidf_word_test1.npy"
)

HT3_qwen06_test = np.load(
    f"{HT3_DIR}/qwen06_lr_test1.npy"
)

HT3_qwen4_test = np.load(
    f"{HT3_DIR}/qwen4_lr_test1.npy"
)

HT3_headtail_test = np.load(
    f"{HT3_DIR}/bert_headtail_optimized/"
    "bert_headtail_optimized_test1.npy"
)

# ------------------------------------------------------------
# 3. Construir matrizes de entrada
# ------------------------------------------------------------

HT3_X_train = np.hstack([
    HT3_word_oof,
    HT3_qwen06_oof,
    HT3_qwen4_oof,
    HT3_headtail_oof
])

HT3_X_test = np.hstack([
    HT3_word_test,
    HT3_qwen06_test,
    HT3_qwen4_test,
    HT3_headtail_test
])

assert HT3_X_train.shape == (20092, 12)
assert HT3_X_test.shape == (900, 12)

# ------------------------------------------------------------
# 4. Carregar rótulos
# ------------------------------------------------------------

HT3_train_df = pd.read_csv(
    "/content/plnEp2/data/train.csv"
)

HT3_test_df = pd.read_excel(
    "/content/plnEp2/data/"
    "test1-rotulado-para-teste-estatistico.xlsx"
)

HT3_y_train = (
    HT3_train_df["clarity"]
    .map(HT3_LABEL2ID)
    .to_numpy()
)

HT3_y_test = (
    HT3_test_df["clarity"]
    .map(HT3_LABEL2ID)
    .to_numpy()
)

assert not pd.isna(HT3_y_train).any()
assert not pd.isna(HT3_y_test).any()

# ------------------------------------------------------------
# 5. Treinar metaclassificador
# ------------------------------------------------------------

HT3_meta_model = LogisticRegression(
    C=1,
    class_weight="balanced",
    max_iter=2000
)

HT3_meta_model.fit(
    HT3_X_train,
    HT3_y_train
)

# ------------------------------------------------------------
# 6. Predições no Teste 1
# ------------------------------------------------------------

HT3_probabilities = HT3_meta_model.predict_proba(
    HT3_X_test
)

HT3_predictions = HT3_meta_model.predict(
    HT3_X_test
)

# ------------------------------------------------------------
# 7. Métricas
# ------------------------------------------------------------

HT3_accuracy = accuracy_score(
    HT3_y_test,
    HT3_predictions
)

HT3_macro_f1 = f1_score(
    HT3_y_test,
    HT3_predictions,
    average="macro"
)

HT3_macro_precision = precision_score(
    HT3_y_test,
    HT3_predictions,
    average="macro",
    zero_division=0
)

HT3_macro_recall = recall_score(
    HT3_y_test,
    HT3_predictions,
    average="macro",
    zero_division=0
)

print("=" * 65)
print("RESULTADOS — STACKING COM BERT HEAD-TAIL OTIMIZADO")
print("=" * 65)

print(f"Accuracy:         {HT3_accuracy:.4f}")
print(f"Macro-F1:         {HT3_macro_f1:.4f}")
print(f"Macro-Precision:  {HT3_macro_precision:.4f}")
print(f"Macro-Recall:     {HT3_macro_recall:.4f}")

# ------------------------------------------------------------
# 8. Relatório por classe
# ------------------------------------------------------------

print("\nRELATÓRIO DE CLASSIFICAÇÃO")
print("=" * 65)

print(
    classification_report(
        HT3_y_test,
        HT3_predictions,
        labels=[0, 1, 2],
        target_names=HT3_CLASSES,
        digits=4,
        zero_division=0
    )
)

print("\nMATRIZ DE CONFUSÃO")
print("=" * 65)

print(
    confusion_matrix(
        HT3_y_test,
        HT3_predictions,
        labels=[0, 1, 2]
    )
)

# ------------------------------------------------------------
# 9. Salvar resultados sem sobrescrever arquivos anteriores
# ------------------------------------------------------------

HT3_metrics = pd.DataFrame([{
    "Accuracy": HT3_accuracy,
    "Macro-F1": HT3_macro_f1,
    "Macro-Precision": HT3_macro_precision,
    "Macro-Recall": HT3_macro_recall
}])

HT3_metrics.to_csv(
    f"{HT3_OUTPUT_DIR}/optimized_stacking_metrics.csv",
    index=False
)

np.save(
    f"{HT3_OUTPUT_DIR}/optimized_stacking_test1_probabilities.npy",
    HT3_probabilities
)

np.save(
    f"{HT3_OUTPUT_DIR}/optimized_stacking_test1_predictions.npy",
    HT3_predictions
)

print("\nResultados salvos em:")
print(HT3_OUTPUT_DIR)

RESULTADOS — STACKING COM BERT HEAD-TAIL OTIMIZADO
Accuracy:         0.5100
Macro-F1:         0.5062
Macro-Precision:  0.5054
Macro-Recall:     0.5100

RELATÓRIO DE CLASSIFICAÇÃO
              precision    recall  f1-score   support

          c1     0.5327    0.5700    0.5507       300
        c234     0.4466    0.3767    0.4087       300
          c5     0.5368    0.5833    0.5591       300

    accuracy                         0.5100       900
   macro avg     0.5054    0.5100    0.5062       900
weighted avg     0.5054    0.5100    0.5062       900


MATRIZ DE CONFUSÃO
[[171  72  57]
 [ 93 113  94]
 [ 57  68 175]]

Resultados salvos em:
/content/ensemble_outputs/stacking_headtail_optimized


#Testando as combinações entre os modelos com o head tail com os novos hiperparametros

In [ ]:
# @title
# ============================================================
# EXPERIMENTOS — COMBINAÇÕES COM BERT HEAD-TAIL OTIMIZADO
#
# 4 modelos:
# 1. TF-IDF Word
# 2. Qwen 0.6B
# 3. Qwen 4B
# 4. BERT Head-Tail otimizado
#
# Testa todas as combinações de 2, 3 e 4 modelos.
# Total: 11 combinações.
#
# Não sobrescreve resultados anteriores.
# Prefixo exclusivo: HTC_
# ============================================================

import os
import json
import itertools
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Configurações
# ------------------------------------------------------------

HTC_BASE_DIR = "/content/ensemble_outputs"

HTC_OUTPUT_DIR = os.path.join(
    HTC_BASE_DIR,
    "combinacoes_headtail_otimizado"
)

os.makedirs(HTC_OUTPUT_DIR, exist_ok=True)

HTC_CLASSES = ["c1", "c234", "c5"]
HTC_LABEL2ID = {
    label: i for i, label in enumerate(HTC_CLASSES)
}

HTC_META_PARAMS = {
    "C": 1,
    "class_weight": "balanced",
    "max_iter": 2000
}

# ------------------------------------------------------------
# 2. Carregar probabilidades OOF e Teste 1
# ------------------------------------------------------------

HTC_models = {
    "TF-IDF Word": {
        "oof": np.load(
            f"{HTC_BASE_DIR}/tfidf_word_oof.npy"
        ),
        "test": np.load(
            f"{HTC_BASE_DIR}/tfidf_word_test1.npy"
        )
    },

    "Qwen 0.6B": {
        "oof": np.load(
            f"{HTC_BASE_DIR}/qwen06_lr_oof.npy"
        ),
        "test": np.load(
            f"{HTC_BASE_DIR}/qwen06_lr_test1.npy"
        )
    },

    "Qwen 4B": {
        "oof": np.load(
            f"{HTC_BASE_DIR}/qwen4_lr_oof.npy"
        ),
        "test": np.load(
            f"{HTC_BASE_DIR}/qwen4_lr_test1.npy"
        )
    },

    "BERT Head-Tail otimizado": {
        "oof": np.load(
            f"{HTC_BASE_DIR}/bert_headtail_optimized/"
            "bert_headtail_optimized_oof.npy"
        ),
        "test": np.load(
            f"{HTC_BASE_DIR}/bert_headtail_optimized/"
            "bert_headtail_optimized_test1.npy"
        )
    }
}

# ------------------------------------------------------------
# 3. Verificar dimensões
# ------------------------------------------------------------

for HTC_name, HTC_data in HTC_models.items():

    assert HTC_data["oof"].shape == (20092, 3), (
        f"Dimensão OOF incorreta: {HTC_name}"
    )

    assert HTC_data["test"].shape == (900, 3), (
        f"Dimensão Teste 1 incorreta: {HTC_name}"
    )

    assert np.isfinite(HTC_data["oof"]).all()
    assert np.isfinite(HTC_data["test"]).all()

print("Todos os modelos carregados e verificados.")

# ------------------------------------------------------------
# 4. Carregar rótulos
# ------------------------------------------------------------

HTC_train_df = pd.read_csv(
    "/content/plnEp2/data/train.csv"
)

HTC_test_df = pd.read_excel(
    "/content/plnEp2/data/"
    "test1-rotulado-para-teste-estatistico.xlsx"
)

HTC_y_train = (
    HTC_train_df["clarity"]
    .map(HTC_LABEL2ID)
    .to_numpy()
)

HTC_y_test = (
    HTC_test_df["clarity"]
    .map(HTC_LABEL2ID)
    .to_numpy()
)

assert not pd.isna(HTC_y_train).any()
assert not pd.isna(HTC_y_test).any()

# ------------------------------------------------------------
# 5. Gerar combinações
# ------------------------------------------------------------

HTC_model_names = list(HTC_models.keys())

HTC_combinations = []

for HTC_size in range(2, len(HTC_model_names) + 1):

    HTC_combinations.extend(
        itertools.combinations(
            HTC_model_names,
            HTC_size
        )
    )

print(f"\nTotal de combinações: {len(HTC_combinations)}")

# ------------------------------------------------------------
# 6. Treinar e avaliar cada combinação
# ------------------------------------------------------------

HTC_results = []
HTC_all_predictions = {}
HTC_all_probabilities = {}

for HTC_idx, HTC_combination in enumerate(
    HTC_combinations,
    start=1
):

    HTC_name = " + ".join(HTC_combination)

    print("\n" + "=" * 75)
    print(f"COMBINAÇÃO {HTC_idx}/{len(HTC_combinations)}")
    print(HTC_name)
    print("=" * 75)

    # Concatenar probabilidades OOF
    HTC_X_train = np.hstack([
        HTC_models[name]["oof"]
        for name in HTC_combination
    ])

    # Concatenar probabilidades do Teste 1
    HTC_X_test = np.hstack([
        HTC_models[name]["test"]
        for name in HTC_combination
    ])

    # Treinar metaclassificador
    HTC_meta = LogisticRegression(
        **HTC_META_PARAMS
    )

    HTC_meta.fit(
        HTC_X_train,
        HTC_y_train
    )

    # Predições
    HTC_probs = HTC_meta.predict_proba(
        HTC_X_test
    )

    HTC_predictions = HTC_meta.predict(
        HTC_X_test
    )

    # Métricas
    HTC_metrics = {
        "Combination": HTC_name,
        "Number_of_models": len(HTC_combination),
        "Accuracy": accuracy_score(
            HTC_y_test,
            HTC_predictions
        ),
        "Macro-F1": f1_score(
            HTC_y_test,
            HTC_predictions,
            average="macro"
        ),
        "Macro-Precision": precision_score(
            HTC_y_test,
            HTC_predictions,
            average="macro",
            zero_division=0
        ),
        "Macro-Recall": recall_score(
            HTC_y_test,
            HTC_predictions,
            average="macro",
            zero_division=0
        )
    }

    HTC_results.append(HTC_metrics)

    # Guardar resultados em memória
    HTC_key = f"combination_{HTC_idx:02d}"

    HTC_all_predictions[HTC_key] = HTC_predictions
    HTC_all_probabilities[HTC_key] = HTC_probs

    print(
        f"Accuracy: {HTC_metrics['Accuracy']:.4f} | "
        f"Macro-F1: {HTC_metrics['Macro-F1']:.4f} | "
        f"Precision: {HTC_metrics['Macro-Precision']:.4f} | "
        f"Recall: {HTC_metrics['Macro-Recall']:.4f}"
    )

    # Salvar probabilidades e predições individualmente
    np.save(
        os.path.join(
            HTC_OUTPUT_DIR,
            f"{HTC_key}_probabilities.npy"
        ),
        HTC_probs
    )

    np.save(
        os.path.join(
            HTC_OUTPUT_DIR,
            f"{HTC_key}_predictions.npy"
        ),
        HTC_predictions
    )

# ------------------------------------------------------------
# 7. Ordenar resultados
# ------------------------------------------------------------

HTC_results_df = pd.DataFrame(HTC_results)

HTC_results_df = HTC_results_df.sort_values(
    by="Macro-F1",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 100)
print("RANKING DAS COMBINAÇÕES — BERT HEAD-TAIL OTIMIZADO")
print("=" * 100)

print(
    HTC_results_df.to_string(
        index=False,
        formatters={
            "Accuracy": "{:.4f}".format,
            "Macro-F1": "{:.4f}".format,
            "Macro-Precision": "{:.4f}".format,
            "Macro-Recall": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# 8. Melhor combinação
# ------------------------------------------------------------

HTC_best = HTC_results_df.iloc[0]

print("\n" + "=" * 75)
print("MELHOR COMBINAÇÃO")
print("=" * 75)

print("Modelos:", HTC_best["Combination"])
print(f"Accuracy: {HTC_best['Accuracy']:.4f}")
print(f"Macro-F1: {HTC_best['Macro-F1']:.4f}")
print(f"Macro-Precision: {HTC_best['Macro-Precision']:.4f}")
print(f"Macro-Recall: {HTC_best['Macro-Recall']:.4f}")

# ------------------------------------------------------------
# 9. Relatório detalhado da melhor combinação
# ------------------------------------------------------------

HTC_best_combination = HTC_best["Combination"].split(" + ")

HTC_best_X_train = np.hstack([
    HTC_models[name]["oof"]
    for name in HTC_best_combination
])

HTC_best_X_test = np.hstack([
    HTC_models[name]["test"]
    for name in HTC_best_combination
])

HTC_best_meta = LogisticRegression(
    **HTC_META_PARAMS
)

HTC_best_meta.fit(
    HTC_best_X_train,
    HTC_y_train
)

HTC_best_predictions = HTC_best_meta.predict(
    HTC_best_X_test
)

HTC_best_probabilities = HTC_best_meta.predict_proba(
    HTC_best_X_test
)

print("\nRELATÓRIO DE CLASSIFICAÇÃO — MELHOR COMBINAÇÃO")
print("=" * 75)

print(
    classification_report(
        HTC_y_test,
        HTC_best_predictions,
        labels=[0, 1, 2],
        target_names=HTC_CLASSES,
        digits=4,
        zero_division=0
    )
)

print("Matriz de confusão:")
print(
    confusion_matrix(
        HTC_y_test,
        HTC_best_predictions,
        labels=[0, 1, 2]
    )
)

# ------------------------------------------------------------
# 10. Salvar ranking e melhor modelo
# ------------------------------------------------------------

HTC_results_df.to_csv(
    os.path.join(
        HTC_OUTPUT_DIR,
        "ranking_combinacoes.csv"
    ),
    index=False
)

np.save(
    os.path.join(
        HTC_OUTPUT_DIR,
        "best_combination_probabilities.npy"
    ),
    HTC_best_probabilities
)

np.save(
    os.path.join(
        HTC_OUTPUT_DIR,
        "best_combination_predictions.npy"
    ),
    HTC_best_predictions
)

HTC_metadata = {
    "models": HTC_model_names,
    "number_of_combinations": len(HTC_combinations),
    "meta_classifier": "LogisticRegression",
    "meta_classifier_parameters": HTC_META_PARAMS,
    "best_combination": HTC_best["Combination"],
    "best_metrics": {
        "accuracy": float(HTC_best["Accuracy"]),
        "macro_f1": float(HTC_best["Macro-F1"]),
        "macro_precision": float(HTC_best["Macro-Precision"]),
        "macro_recall": float(HTC_best["Macro-Recall"])
    }
}

with open(
    os.path.join(
        HTC_OUTPUT_DIR,
        "experiment_metadata.json"
    ),
    "w"
) as f:
    json.dump(HTC_metadata, f, indent=4)

print("\n" + "=" * 75)
print("EXPERIMENTOS CONCLUÍDOS")
print("=" * 75)

print("Arquivos salvos em:")
print(HTC_OUTPUT_DIR)

print("Nenhum resultado anterior foi sobrescrito.")

Todos os modelos carregados e verificados.

Total de combinações: 11

COMBINAÇÃO 1/11
TF-IDF Word + Qwen 0.6B
Accuracy: 0.5044 | Macro-F1: 0.4983 | Precision: 0.4993 | Recall: 0.5044

COMBINAÇÃO 2/11
TF-IDF Word + Qwen 4B
Accuracy: 0.5067 | Macro-F1: 0.5020 | Precision: 0.5016 | Recall: 0.5067

COMBINAÇÃO 3/11
TF-IDF Word + BERT Head-Tail otimizado
Accuracy: 0.5000 | Macro-F1: 0.4972 | Precision: 0.4961 | Recall: 0.5000

COMBINAÇÃO 4/11
Qwen 0.6B + Qwen 4B
Accuracy: 0.4989 | Macro-F1: 0.4937 | Precision: 0.4944 | Recall: 0.4989

COMBINAÇÃO 5/11
Qwen 0.6B + BERT Head-Tail otimizado
Accuracy: 0.4911 | Macro-F1: 0.4856 | Precision: 0.4854 | Recall: 0.4911

COMBINAÇÃO 6/11
Qwen 4B + BERT Head-Tail otimizado
Accuracy: 0.4856 | Macro-F1: 0.4809 | Precision: 0.4797 | Recall: 0.4856

COMBINAÇÃO 7/11
TF-IDF Word + Qwen 0.6B + Qwen 4B
Accuracy: 0.5056 | Macro-F1: 0.5006 | Precision: 0.5005 | Recall: 0.5056

COMBINAÇÃO 8/11
TF-IDF Word + Qwen 0.6B + BERT Head-Tail otimizado
Accuracy: 0.5056 | Mac

#Raciocinio sobre o Grid Search
Raciocinando sobre os resultados "piores" do Bert head-tail com os hiperparametros escolhidos via Grid Search::::

O grid search encontrou uma configuração que teve melhor desempenho na validação 80/20, mas isso não necessariamente significa que ela produzirá probabilidades melhores para o stacking.

Vamos segir com a configuração anterior

==> lembrando que da Reg Log fizemos no ep1

#Salvando os resultados ("seletivamente")
o modelo final é o stacking com quatro componentes, não o conjunto de todos os modelos que vocês experimentaram:
1. TF-IDF Word + Logistic Regression — a configuração do modelo do EP1, reutilizada como componente.
2. Qwen3-Embedding-0.6B + Logistic Regression.
3. Qwen3-Embedding-4B + Logistic Regression.
4. BERTimbau Base Head-Tail "original".

O meta-classificador combina as probabilidades desses quatro componentes por meio de Logistic Regression. Isso gera 12 features de entrada (3 probabilidades por modelo).


O BERT Head-Tail "otimizado" pelo Grid Search não deve substituir o original: nos experimentos a versão otimizada não melhorou o resultado da combinação. Portanto, estou compartilhando os resultados mantendo o Head-Tail "original" como parte do modelo final.

In [27]:
# @title Finalização da exportação — Modelo Final EP2

import os
import json
import pandas as pd

# ============================================================
# 1. DIRETÓRIOS
# ============================================================

DEST_DIR = "/content/plnEp2/modelo_final"

PROB_DIR = os.path.join(DEST_DIR, "probabilidades")
MODEL_DIR = os.path.join(DEST_DIR, "modelos")
EMBED_DIR = os.path.join(DEST_DIR, "embeddings")
VALID_DIR = os.path.join(DEST_DIR, "validacao")
META_DIR = os.path.join(DEST_DIR, "metadados")

for directory in [
    PROB_DIR, MODEL_DIR, EMBED_DIR, VALID_DIR, META_DIR
]:
    os.makedirs(directory, exist_ok=True)


# ============================================================
# 2. RESULTADOS DA EXECUÇÃO ANTERIOR DO NESTED STACKING
# ============================================================

# Resultados históricos de 5 folds externos.
# Não representam uma nova execução nesta sessão.

fold_results = pd.DataFrame({
    "fold": [1, 2, 3, 4, 5],
    "accuracy": [0.4747, 0.4882, 0.4741, 0.4863, 0.4749],
    "macro_f1": [0.4693, 0.4803, 0.4662, 0.4810, 0.4701],
    "macro_precision": [0.4695, 0.4801, 0.4654, 0.4818, 0.4696],
    "macro_recall": [0.4762, 0.4899, 0.4754, 0.4883, 0.4762]
})

fold_results.to_csv(
    os.path.join(VALID_DIR, "nested_stacking_fold_results.csv"),
    index=False
)

nested_summary = pd.DataFrame({
    "metric": [
        "accuracy",
        "macro_f1",
        "macro_precision",
        "macro_recall"
    ],
    "mean": [
        0.4796,
        0.4734,
        0.4733,
        0.4812
    ],
    "std": [
        0.0070,
        0.0068,
        0.0072,
        0.0072
    ]
})

nested_summary.to_csv(
    os.path.join(VALID_DIR, "nested_stacking_summary.csv"),
    index=False
)

# Resultados agrupados das predições OOF externas.
nested_pooled = pd.DataFrame({
    "metric": [
        "accuracy",
        "macro_f1",
        "macro_precision",
        "macro_recall"
    ],
    "value": [
        0.4796,
        0.4735,
        0.4733,
        0.4812
    ]
})

nested_pooled.to_csv(
    os.path.join(VALID_DIR, "nested_stacking_pooled_metrics.csv"),
    index=False
)

# Documentação da origem dos resultados.
nested_metadata = {
    "experiment": "Nested Stacking",
    "outer_folds": 5,
    "inner_folds": 4,
    "status": "Resultados de execução anterior",
    "raw_predictions_included": False,
    "raw_fold_probabilities_included": False,
    "note": (
        "Este diretório contém somente os resultados agregados "
        "e as métricas por fold. Os arrays originais de predições "
        "e probabilidades não foram recuperados nesta exportação."
    )
}

with open(
    os.path.join(VALID_DIR, "nested_stacking_metadata.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(nested_metadata, f, ensure_ascii=False, indent=4)


# ============================================================
# 3. METADADOS DO MODELO FINAL
# ============================================================

metadata = {
    "project": "EP2 - Introdução ao Processamento de Língua Natural",
    "final_architecture": "Stacking",

    "base_models": [
        {
            "name": "TF-IDF Word + Logistic Regression",
            "vectorizer": {
                "analyzer": "word",
                "ngram_range": [1, 3],
                "sublinear_tf": True,
                "lowercase": True
            },
            "classifier": {
                "C": 1.0,
                "class_weight": "balanced",
                "max_iter": 2000
            }
        },
        {
            "name": "Qwen3-Embedding-0.6B + Logistic Regression",
            "model_id": "Qwen/Qwen3-Embedding-0.6B",
            "embedding_dimension": 1024,
            "classifier": {
                "C": 1.0,
                "class_weight": "balanced",
                "max_iter": 2000
            }
        },
        {
            "name": "Qwen3-Embedding-4B + Logistic Regression",
            "model_id": "Qwen/Qwen3-Embedding-4B",
            "embedding_dimension": 2560,
            "classifier": {
                "C": 1.0,
                "class_weight": "balanced",
                "max_iter": 2000
            }
        },
        {
            "name": "BERTimbau Base Head-Tail original",
            "model_id": "neuralmind/bert-base-portuguese-cased",
            "learning_rate": BERT_LR,
            "epochs": BERT_EPOCHS,
            "weight_decay": BERT_WEIGHT_DECAY,
            "max_length": BERT_MAX_LENGTH,
            "head_tokens": 127,
            "tail_tokens": 127
        }
    ],

    "meta_classifier": {
        "name": "LogisticRegression",
        "C": 1.0,
        "class_weight": "balanced",
        "max_iter": 2000,
        "input_features": 12,
        "feature_order": [
            "TFIDF_Word",
            "Qwen_0.6B",
            "Qwen_4B",
            "BERT_HeadTail"
        ]
    },

    "classes": list(CLASSES),

    "test2_predictions": (
        "Não incluídas. Serão geradas por outro integrante."
    )
}

with open(
    os.path.join(META_DIR, "modelo_final_metadata.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4)


# ============================================================
# 4. README
# ============================================================

readme = """
# Modelo Final — EP2

## Arquitetura

Stacking composto por quatro modelos-base:

1. TF-IDF Word + Logistic Regression
2. Qwen3-Embedding-0.6B + Logistic Regression
3. Qwen3-Embedding-4B + Logistic Regression
4. BERTimbau Base Head-Tail original

As probabilidades dos quatro modelos são concatenadas,
produzindo 12 features para um meta-classificador
Logistic Regression.

## Estrutura

- probabilidades/: probabilidades OOF e Teste 1.
- modelos/: modelos-base e meta-classificador.
- embeddings/: embeddings extraídos pelos Qwen.
- validacao/: resultados agregados do Nested Stacking.
- metadados/: configurações e rótulos codificados.

## Validação

Os resultados do Nested Stacking correspondem a uma execução
anterior com 5 folds externos e 4 folds internos.

Foram preservadas as métricas por fold e as métricas agregadas.
Os arrays originais de predições e probabilidades dessa execução
não estão incluídos.

## Observações

- O Teste 1 é utilizado para avaliação e comparação estatística
  com o EP1.
- O Teste 2 é reservado para a avaliação oficial do EP2.
- As predições do Teste 2 serão geradas separadamente.

## Ordem das features do meta-classificador

1. TF-IDF Word: c1, c234, c5
2. Qwen 0.6B: c1, c234, c5
3. Qwen 4B: c1, c234, c5
4. BERT Head-Tail: c1, c234, c5
"""

with open(
    os.path.join(DEST_DIR, "README.md"),
    "w",
    encoding="utf-8"
) as f:
    f.write(readme)


# ============================================================
# 5. INVENTÁRIO FINAL
# ============================================================

inventory = []

for root, dirs, files in os.walk(DEST_DIR):
    for filename in files:

        path = os.path.join(root, filename)
        size = os.path.getsize(path)

        inventory.append({
            "file": filename,
            "path": os.path.relpath(path, DEST_DIR),
            "size_bytes": size,
            "size_MB": round(size / (1024 ** 2), 2)
        })

inventory_df = pd.DataFrame(inventory).sort_values("path")

inventory_df.to_csv(
    os.path.join(DEST_DIR, "artifact_inventory.csv"),
    index=False
)

print("=" * 65)
print("EXPORTAÇÃO FINALIZADA")
print("=" * 65)

print("Destino:", DEST_DIR)
print("Total de arquivos:", len(inventory_df))

print(
    "Tamanho total:",
    round(inventory_df["size_bytes"].sum() / (1024 ** 3), 2),
    "GB"
)

display(inventory_df)

EXPORTAÇÃO FINALIZADA
Destino: /content/plnEp2/modelo_final
Total de arquivos: 32
Tamanho total: 0.77 GB


,file,path,size_bytes,size_MB
0,README.md,README.md,1315,0.00
31,qwen06_test1_embeddings.npy,embeddings/qwen06_test1_embeddings.npy,3686528,3.52
30,qwen06_train_embeddings.npy,embeddings/qwen06_train_embeddings.npy,82296960,78.48
28,qwen4_test1_embeddings.npy,embeddings/qwen4_test1_embeddings.npy,9216128,8.79
29,qwen4_train_embeddings.npy,embeddings/qwen4_train_embeddings.npy,205742208,196.21
7,modelo_final_metadata.json,metadados/modelo_final_metadata.json,2059,0.00
5,y_test1_encoded.npy,metadados/y_test1_encoded.npy,7328,0.01
6,y_train_encoded.npy,metadados/y_train_encoded.npy,160864,0.15
14,config.json,modelos/bert_headtail_final_model/config.json,988,0.00
15,model.safetensors,modelos/bert_headtail_final_model/model.safete...,435725300,415.54


#Comparação com o modelo do EP1

# Geração das predições para o Teste 2